# 🩺 AI-Powered Diabetic Retinopathy Grading System
## Retinal Fundus Image Classification Using Deep Learning
### EfficientNetV2-S + Grad-CAM++ | High-Resolution 1024px | APTOS 2019

> ⚠️ **RESEARCH USE ONLY** — Not approved for clinical deployment.

| Upgrade | Detail |
|---------|--------|
| Backbone | EfficientNetV2-S (21M params, ImageNet top-1 ~84.9%) |
| Resolution | Progressive 256 → 512 → 1024 px |
| Training | Mixed Precision (AMP) + Gradient Accumulation |
| Augmentation | RandAugment + Medical CLAHE + 360° Rotation |
| Loss | 0.5×CE(label_smooth=0.1) + 0.5×Focal(γ=2) |
| Workers | num_workers=2, pin_memory=True, persistent_workers=True |

## ⚙️ Step 1 — Install & Import Requirements

In [ ]:
%%capture
# Core ML
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q timm>=0.9.0
!pip install -q albumentations>=1.3.1
!pip install -q opencv-python-headless>=4.9.0
!pip install -q scikit-learn>=1.3.0
!pip install -q scikit-image>=0.21.0
!pip install -q scipy>=1.11.0
!pip install -q matplotlib>=3.7.0
!pip install -q tqdm>=4.66.0
!pip install -q grad-cam>=1.5.2
!pip install -q pyyaml>=6.0.1
!pip install -q gradio>=4.0.0
!pip install -q kaggle
!pip install -q gunicorn>=21.2.0
print('✅ All packages installed.')

In [ ]:
from pytorch_grad_cam import GradCAMPlusPlus
from pytorch_grad_cam.utils.image import show_cam_on_image
print("✅ pytorch-grad-cam modules imported successfully.")

In [ ]:
import os, sys, io, json, gc, time, random, shutil, warnings, zipfile, pickle
from pathlib import Path
from copy import deepcopy
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2
from PIL import Image
from tqdm.auto import tqdm
import yaml
import scipy.stats as stats

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
import torchvision.transforms as T

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_auc_score, average_precision_score,
    cohen_kappa_score, ConfusionMatrixDisplay
)

from pytorch_grad_cam import GradCAMPlusPlus
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

warnings.filterwarnings('ignore')

# ── Colab guard ───────────────────────────────────────────────────────────────
try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False; colab_files = None

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

seed_everything()
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✅ Imports complete. Device: {DEVICE.upper()}')
print(f'   PyTorch {torch.__version__} | timm {timm.__version__}')
print(f'   Running in Colab: {IN_COLAB}')


## 🔑 Step 2 — Kaggle API Setup

In [ ]:
# ── Kaggle credential setup (Colab upload OR environment variables) ──────────
kaggle_dir  = Path.home() / ".kaggle"
kaggle_json = kaggle_dir / "kaggle.json"
kaggle_dir.mkdir(exist_ok=True)

if not kaggle_json.exists():
    # 1) Try environment variables (works on Render / local / CI)
    u = os.environ.get("KAGGLE_USERNAME")
    k = os.environ.get("KAGGLE_KEY")
    if u and k:
        kaggle_json.write_text(json.dumps({"username": u, "key": k}))
        os.chmod(kaggle_json, 0o600)
        print(f"✅ Kaggle credentials set from environment variables.")
    elif IN_COLAB:
        # 2) Interactive upload in Colab
        print("📂 Please upload your kaggle.json file:")
        uploaded = colab_files.upload()
        for fname, content in uploaded.items():
            kaggle_json.write_bytes(content)
            os.chmod(kaggle_json, 0o600)
            print(f"✅ Saved to {kaggle_json}")
    else:
        print("⚠️  No Kaggle credentials found.")
        print("   Set KAGGLE_USERNAME and KAGGLE_KEY env vars, or place kaggle.json at ~/.kaggle/kaggle.json")
else:
    print(f"✅ kaggle.json already present at {kaggle_json}")

!kaggle --version

## 📥 Step 3 — Dataset Download (APTOS 2019)

In [ ]:
# DATA_DIR can be overridden via env var for local / Render usage
DATA_DIR = Path(os.environ.get("DATA_DIR", "/content/aptos2019"))
DATA_DIR.mkdir(parents=True, exist_ok=True)

zip_path = DATA_DIR / "aptos2019-blindness-detection.zip"

if not zip_path.exists():
    print("⬇️  Downloading APTOS 2019 from Kaggle (~350 MB)...")
    !kaggle competitions download -c aptos2019-blindness-detection -p {DATA_DIR}
    print("✅ Download complete.")
else:
    print(f"✅ Zip already exists at {zip_path}")

## 📦 Step 4 — Dataset Extraction

In [ ]:
IMG_DIR  = DATA_DIR / "train_images"
CSV_PATH = DATA_DIR / "train.csv"

if not IMG_DIR.exists() or not CSV_PATH.exists():
    print("📦 Extracting archive...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(DATA_DIR)
    print("✅ Extraction complete.")
else:
    print(f"✅ Images: {IMG_DIR} | CSV: {CSV_PATH}")

imgs = list(IMG_DIR.glob("*.png"))
print(f"   Total images found: {len(imgs):,}")
print(f"   CSV shape: {pd.read_csv(CSV_PATH).shape}")


## 🏷️ Step 5 — Data Labeling & Path Mapping

In [ ]:
df = pd.read_csv(CSV_PATH)
df["image_path"] = df["id_code"].apply(lambda x: str(IMG_DIR / f"{x}.png"))
df["file_exists"] = df["image_path"].apply(lambda p: Path(p).exists())

# ── DR Grade labels ────────────────────────────────────────────────────────────
GRADE_MAP = {
    0: "No DR",
    1: "Mild DR",
    2: "Moderate DR",
    3: "Severe DR",
    4: "Proliferative DR (PDR)"
}
GRADE_COLORS = ["#2ecc71","#f1c40f","#e67e22","#e74c3c","#8e44ad"]

df["grade_label"] = df["diagnosis"].map(GRADE_MAP)

# Binary label for ROC/PR (clinical: referable = grade ≥ 2)
df["binary"] = (df["diagnosis"] >= 2).astype(int)

display(df[["id_code","diagnosis","grade_label","binary","file_exists"]].head(10))
print(f"\n✅ Total: {len(df):,} | Missing files: {(~df['file_exists']).sum()}")

## 📊 Step 6 — Exploratory Data Analysis (EDA)

In [ ]:
# ── Class Distribution ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

counts = df["diagnosis"].value_counts().sort_index()
bars = axes[0].bar([GRADE_MAP[i] for i in counts.index], counts.values,
                    color=GRADE_COLORS, edgecolor="black", linewidth=0.8)
axes[0].set_title("DR Grade Distribution (Multi-class)", fontsize=13, fontweight="bold")
axes[0].set_xlabel("DR Grade"); axes[0].set_ylabel("Count")
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                 f"{val}\n({val/len(df)*100:.1f}%)", ha="center", fontsize=9)
axes[0].tick_params(axis='x', rotation=30)

# Binary distribution
bin_counts = df["binary"].value_counts().sort_index()
wedges, texts, autotexts = axes[1].pie(
    bin_counts, labels=["Non-Referable (0–1)", "Referable DR (≥2)"],
    autopct="%1.1f%%", colors=["#2ecc71","#e74c3c"],
    startangle=90, explode=(0, 0.05))
axes[1].set_title("Binary Classification Split\n(Clinical Screening Threshold)", fontsize=13, fontweight="bold")
for at in autotexts: at.set_fontsize(12)

plt.tight_layout(); plt.savefig("/content/eda_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Class counts:\n{counts.to_string()}")
print(f"\nImbalance ratio: {counts.max()/counts.min():.1f}x")


In [ ]:
# ── Sample Images Grid ─────────────────────────────────────────────────────────
n_per_class = 3
fig, axes = plt.subplots(5, n_per_class, figsize=(12, 20))

for grade in range(5):
    sample = df[df["diagnosis"] == grade].sample(n_per_class, random_state=SEED)
    for j, (_, row) in enumerate(sample.iterrows()):
        img = cv2.imread(row["image_path"])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (224, 224))
        ax = axes[grade][j]
        ax.imshow(img); ax.axis("off")
        if j == 0:
            ax.set_ylabel(f"Grade {grade}\n{GRADE_MAP[grade]}",
                          fontsize=10, fontweight="bold",
                          color=GRADE_COLORS[grade], rotation=0,
                          labelpad=80, va="center")

plt.suptitle("Sample Fundus Images by DR Grade (Raw)", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("/content/eda_samples.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ── Image Statistics ──────────────────────────────────────────────────────────
print("📐 Image Size Analysis (sampling 100 images)...")
sample_paths = df["image_path"].sample(100, random_state=SEED).tolist()
sizes = []
for p in tqdm(sample_paths):
    img = cv2.imread(p)
    if img is not None:
        sizes.append(img.shape[:2])  # (H, W)

heights, widths = zip(*sizes)
print(f"   Height: min={min(heights)}, max={max(heights)}, mean={np.mean(heights):.0f}")
print(f"   Width:  min={min(widths)}, max={max(widths)}, mean={np.mean(widths):.0f}")
print(f"   Unique sizes: {len(set(sizes))}")


## 🧹 Step 7 — Data Cleaning

In [ ]:
import time as _time
t0 = _time.time()

print(f'Initial rows: {len(df):,}')

# ── Step A: vectorised file-existence check (instant) ─────────────────────────
df['file_exists'] = df['image_path'].apply(lambda p: os.path.isfile(p))
missing = (~df['file_exists']).sum()
df = df[df['file_exists']].copy().reset_index(drop=True)
print(f'After removing missing files : {len(df):,}  (removed {missing})')

# ── Step B: parallel corrupt-image check ─────────────────────────────────────
# Uses IMREAD_REDUCED_GRAYSCALE_2 — reads only 1/4 of pixels for fast decode test.
def _check_fast(path):
    try:
        raw = open(str(path), 'rb').read()
        img = cv2.imdecode(np.frombuffer(raw, np.uint8),
                           cv2.IMREAD_REDUCED_GRAYSCALE_2)
        return img is not None and img.size >= 625  # ≥50×50 at half-size
    except Exception:
        return False

WORKERS = min(os.cpu_count() or 2, 4)  # Colab typically has 2 physical cores
paths   = df['image_path'].tolist()
with ThreadPoolExecutor(max_workers=WORKERS) as ex:
    valid_flags = list(tqdm(ex.map(_check_fast, paths), total=len(df),
                            desc='⚡ Integrity check'))

df['valid'] = valid_flags
bad = (~df['valid']).sum()
if bad:
    print(f'⚠️  Corrupted / unreadable: {bad}')
    print(df[~df['valid']][['id_code','diagnosis']].to_string())
df = df[df['valid']].reset_index(drop=True)

print(f'\nClean dataset: {len(df):,} images  |  elapsed {_time.time()-t0:.1f}s')
print(df['diagnosis'].value_counts().sort_index().to_string())
gc.collect()


## 🖼️ Step 8 — Data Preprocessing (Ben Graham's Method + Circular Cropping + CLAHE)

In [ ]:
# ── Resolution config ────────────────────────────────────────────────────────
# 512 = recommended sweet-spot for speed vs quality.
# 1024+ supported: set IMG_SIZE env var; gradient-checkpointing auto-enables.
IMG_SIZE = int(os.environ.get('IMG_SIZE', 512))
print(f'📐 Input resolution: {IMG_SIZE}×{IMG_SIZE} px')

# ── Precompute sigma (odd, avoid recompute per image) ────────────────────────
_BG_SIGMA = max((IMG_SIZE // 10) | 1, 1)   # odd integer

# ── Internal helpers — all operate on ALREADY-RESIZED square image ───────────
def _make_mask(rgb_sq):
    """Fast retina circle mask on a pre-sized square RGB image."""
    g      = rgb_sq[:, :, 1]                        # green channel
    gb     = cv2.medianBlur(g, 7)
    _, th  = cv2.threshold(gb, 0, 255,
                           cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    th     = cv2.morphologyEx(th, cv2.MORPH_OPEN,
                               cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(7,7)))
    cnts, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return np.ones(g.shape, np.uint8) * 255
    c    = max(cnts, key=cv2.contourArea)
    mask = np.zeros_like(g, np.uint8)
    cv2.drawContours(mask, [c], -1, 255, -1)
    (cx, cy), r = cv2.minEnclosingCircle(c)
    circ = np.zeros_like(g, np.uint8)
    cv2.circle(circ, (int(cx), int(cy)), int(r * 0.97), 255, -1)
    return cv2.bitwise_and(mask, circ)

def _clahe_lab(rgb):
    lab       = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
    l, a, b   = cv2.split(lab)
    l2        = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8)).apply(l)
    return cv2.cvtColor(cv2.merge([l2, a, b]), cv2.COLOR_LAB2RGB)

def _green_emphasis(rgb):
    """Emphasise green channel in-place (no extra allocation)."""
    r, g, b = rgb[:,:,0].astype(np.float32), \
              rgb[:,:,1].astype(np.float32), \
              rgb[:,:,2].astype(np.float32)
    mix = (r * 0.1 + g * 0.8 + b * 0.1).clip(0, 255).astype(np.uint8)
    out = np.stack([mix, rgb[:,:,1], mix], axis=2)
    return out

# ── PUBLIC API ────────────────────────────────────────────────────────────────
def preprocess_fundus(path, size=None):
    """
    FAST fundus preprocessing (resize-first strategy).
    ~8× faster than process-at-native-resolution approach.

    Strategy:
      1. Read → resize to target SIZE immediately (all ops on small image)
      2. Pad to square
      3. Compute retina mask ONCE
      4. CLAHE → Ben Graham → re-mask → green emphasis
    """
    if size is None: size = IMG_SIZE
    bgr = cv2.imread(str(path))
    if bgr is None: return None
    h, w   = bgr.shape[:2]
    rgb    = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

    # ── 1. Aspect-preserving resize to target (INTER_AREA = fastest + best quality)
    s      = size / max(h, w)
    nh, nw = int(round(h * s)), int(round(w * s))
    rgb    = cv2.resize(rgb, (nw, nh), interpolation=cv2.INTER_AREA)

    # ── 2. Reflection-pad to square
    pt = (size - nh) // 2;  pb = size - nh - pt
    pl = (size - nw) // 2;  pr = size - nw - pl
    rgb = cv2.copyMakeBorder(rgb, pt, pb, pl, pr, cv2.BORDER_REFLECT_101)

    # ── 3. Single mask computation on small image
    mask = _make_mask(rgb)
    rgb[mask == 0] = 0

    # ── 4. CLAHE
    rgb = _clahe_lab(rgb)

    # ── 5. Ben Graham (local average subtraction)
    sig  = _BG_SIGMA if size == IMG_SIZE else max((size // 10) | 1, 1)
    blur = cv2.GaussianBlur(rgb, (0, 0), sigmaX=sig)
    rgb  = cv2.addWeighted(rgb, 4, blur, -4, 128)
    rgb[mask == 0] = 0        # re-apply mask (one reuse, no recompute)

    # ── 6. Green-channel emphasis
    rgb = _green_emphasis(rgb)
    return rgb

print(f'✅ Fast preprocessing defined  (sigma={_BG_SIGMA}, resize-first strategy).')

# ── Latency sanity check on one sample ───────────────────────────────────────
_sample = df['image_path'].iloc[0]
_t = time.time()
for _ in range(5): preprocess_fundus(_sample)
print(f'   Per-image latency: {(time.time()-_t)/5*1000:.1f} ms  (target <50 ms)')


In [ ]:
# ── Visualize preprocessing stages ───────────────────────────────────────────
sample_paths_per_grade = {g: df[df["diagnosis"]==g]["image_path"].iloc[0]
                          for g in range(5)}
stages = [
    ("Raw",             lambda p: cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)),
    ("Circular Mask",   lambda p: (lambda rgb: (rgb.__setitem__(slice(None), cv2.bitwise_and(rgb, cv2.merge([make_retina_mask(rgb)]*3))), rgb)[1])(cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB))),
    ("+ CLAHE",         lambda p: apply_clahe(cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB))),
    ("Full Pipeline",   lambda p: preprocess_fundus(p, use_green=False)),
    ("+ Green Emph.",   lambda p: preprocess_fundus(p)),
]

fig, axes = plt.subplots(5, len(stages), figsize=(18, 20))
for gi in range(5):
    for si, (stage_name, fn) in enumerate(stages):
        try:
            img = fn(sample_paths_per_grade[gi])
            if img is not None:
                img_show = cv2.resize(img, (224,224))
                axes[gi][si].imshow(img_show)
        except:
            pass
        axes[gi][si].axis("off")
        if gi == 0:
            axes[gi][si].set_title(stage_name, fontsize=11, fontweight="bold", pad=8)
    axes[gi][0].set_ylabel(f"Grade {gi}\n{GRADE_MAP[gi]}", fontsize=9,
                            fontweight="bold", color=GRADE_COLORS[gi],
                            rotation=0, labelpad=90, va="center")

plt.suptitle("Preprocessing Pipeline — All DR Grades", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("/content/preprocessing_stages.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Preprocessing visualization complete.")


## 🛡️ Step 8b — Multilayer Fundus Image Verifier (One-Class Anomaly Detection)

> Ensures **only valid retinal fundus images** reach the DR model.
> Non-fundus images (X-rays, photos, cartoons, noise) are **blocked** with a warning.

### Pipeline (5 stages, ~80–150 ms total)
| Stage | Method | Purpose |
|-------|--------|---------|
| 1 | Color statistics (numpy) | Green dominance, orange-red hue, brightness |
| 2 | Blood vessel analysis (morphology + top-hat) | Branching vasculature, vessel density |
| 3 | Vessel continuity (connected components) | Tree-like topology, spatial distribution |
| 4 | Deep embedding (MobileNetV3-Small → PCA) | Learned fundus feature distribution |
| 5 | Anomaly gate (IsolationForest) | One-class OOD detection against APTOS |

**Design principles:**
- Does **NOT** rely on circular shape (handles cropped/zoomed/partial images)
- Trained **only on APTOS** — no non-fundus examples needed (one-class)
- Explicit vessel network integrity check (not just edge density)
- Fast-reject path skips heavy stages for obvious non-fundus inputs


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# MULTILAYER FUNDUS VERIFIER — Class Definition
# ═══════════════════════════════════════════════════════════════════════════

class FundusVerifier:
    """
    5-stage one-class fundus image verifier.
    Trained purely on APTOS 2019 fundus images via anomaly detection.
    No non-fundus training data required.
    """

    # ── internal transform for feature extractor ──────────────────────────
    _INF_SIZE = 224
    _MEAN     = [0.485, 0.456, 0.406]
    _STD      = [0.229, 0.224, 0.225]

    def __init__(self, device=DEVICE, n_pca=64, threshold=0.38):
        self.device    = device
        self.n_pca     = n_pca
        self.threshold = threshold   # confidence below this → reject
        self.fitted    = False

        # ── Lightweight feature extractor (MobileNetV3-Small, ~1.5M params) ─
        self._extractor = timm.create_model(
            'mobilenetv3_small_050', pretrained=True, num_classes=0
        ).to(device).eval()
        self._tfm = A.Compose([
            A.Resize(self._INF_SIZE, self._INF_SIZE),
            A.Normalize(mean=self._MEAN, std=self._STD),
            ToTensorV2()
        ])

        # Fitted attributes
        self.pca = self.clf = None
        self._score_mu = self._score_sg = 0.0
        self._s1_mu    = self._s1_sg    = 0.0

    # ──────────────────────────────────────────────────────────────────────
    # STAGE 1 — Visual color screening (~0.5 ms)
    # ──────────────────────────────────────────────────────────────────────
    def _stage1_visual(self, rgb):
        arr = rgb.astype(np.float32)
        r, g, b = arr[:,:,0], arr[:,:,1], arr[:,:,2]

        # a) Green dominance (retinal images: green channel strongest)
        g_dom = float(g.mean() / (r.mean() + b.mean() + 1e-6))

        # b) Orange-red hue fraction (fundus characteristic)
        hsv  = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV).astype(np.float32)
        hue  = hsv[:,:,0]
        vmask = hsv[:,:,2] > 20
        if vmask.sum() > 100:
            h_vals   = hue[vmask]
            or_red   = float(((h_vals < 30) | (h_vals > 155)).mean())
        else:
            or_red = 0.0

        # c) Brightness validity
        g_mean = float(g.mean())
        bright_ok = float(8.0 < g_mean < 235.0)

        # d) Dark-border ratio (fundus images have significant black border)
        dark_r = float((g < 12).mean())
        border_ok = float(dark_r < 0.80)   # not almost all black

        score = (
            min(g_dom / 1.4, 1.0) * 0.30 +
            or_red                * 0.30 +
            bright_ok             * 0.20 +
            border_ok             * 0.20
        )
        return float(score), {'g_dom': g_dom, 'or_red': or_red,
                              'g_mean': g_mean, 'dark_ratio': dark_r}

    # ──────────────────────────────────────────────────────────────────────
    # STAGE 2+3 — Blood vessel pattern analysis (~8 ms on 256×256)
    # ──────────────────────────────────────────────────────────────────────
    def _stage23_vessels(self, rgb):
        """Detect retinal vasculature via multi-scale morphological ridge detection.
        Explicitly avoids circular-shape heuristics.
        Checks: branching topology, tree-like continuity, spatial distribution.
        """
        small = cv2.resize(rgb, (256, 256), interpolation=cv2.INTER_AREA)
        g     = small[:,:,1].copy()

        # CLAHE on green channel (enhances dark vessels on bright background)
        g_eq  = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8)).apply(g)

        # Multi-scale black top-hat (detects dark thin vessels)
        vessel_resp = np.zeros_like(g_eq, np.float32)
        for ks in [7, 11, 15, 21]:   # multiple scales → branching at different widths
            k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (ks, ks))
            bth = cv2.morphologyEx(g_eq, cv2.MORPH_BLACKHAT, k).astype(np.float32)
            vessel_resp = np.maximum(vessel_resp, bth)

        # Normalise
        v_min, v_max = vessel_resp.min(), vessel_resp.max()
        if v_max > v_min:
            vessel_norm = (vessel_resp - v_min) / (v_max - v_min)
        else:
            vessel_norm = vessel_resp

        # Threshold → binary vessel map
        vessel_bin = (vessel_norm > 0.18).astype(np.uint8) * 255

        # ── Stage 2: density & response strength ─────────────────────────
        vessel_density  = float(vessel_bin.mean()) / 255.0
        mean_response   = float(vessel_norm.mean())

        # ── Stage 3: connectivity / branching topology ────────────────────
        n_lbls, lbl_img, stats, _ = cv2.connectedComponentsWithStats(
            vessel_bin, connectivity=8)
        if n_lbls > 1:
            areas = stats[1:, cv2.CC_STAT_AREA]
            # Tree-like vasculature: several large + many small components
            large_comps  = int((areas >= 20).sum())
            small_comps  = int((areas < 20).sum())
            max_frac     = float(areas.max() / (256*256))
        else:
            large_comps = small_comps = 0; max_frac = 0.0

        # Spatial distribution: divide into 4 quadrants, vessels in ≥2 = real retina
        q_scores = []
        for r0, r1, c0, c1 in [(0,128,0,128),(0,128,128,256),
                                (128,256,0,128),(128,256,128,256)]:
            q_scores.append(float(vessel_bin[r0:r1,c0:c1].mean())/255)
        n_active_quads = sum(q > 0.005 for q in q_scores)

        # ── Composite score ───────────────────────────────────────────────
        # Vessel density: real fundus has 2-15% vessel pixels
        density_ok  = float(0.005 < vessel_density < 0.20)
        response_ok = min(mean_response / 0.05, 1.0)
        branch_ok   = min(large_comps / 4.0, 1.0)
        spread_ok   = n_active_quads / 4.0

        score = (
            density_ok  * 0.25 +
            response_ok * 0.25 +
            branch_ok   * 0.25 +
            spread_ok   * 0.25
        )
        return float(score), {
            'vessel_density': round(vessel_density, 4),
            'mean_response':  round(mean_response, 4),
            'large_comps':    large_comps,
            'n_active_quads': n_active_quads,
        }

    # ──────────────────────────────────────────────────────────────────────
    # STAGE 4+5 — Deep embedding + IsolationForest anomaly detection
    # ──────────────────────────────────────────────────────────────────────
    def _extract_features(self, rgb):
        t   = self._tfm(image=rgb)['image'].unsqueeze(0).to(self.device)
        with torch.no_grad():
            return self._extractor(t).cpu().numpy().flatten()

    def _stage45_deep(self, rgb):
        if not self.fitted:
            return 0.5, {}
        feat  = self._extract_features(rgb).reshape(1, -1)
        xpca  = self.pca.transform(feat)
        raw   = float(self.clf.score_samples(xpca)[0])
        z     = (raw - self._score_mu) / (self._score_sg + 1e-6)
        score = float(1.0 / (1.0 + np.exp(-z * 2.5)))   # sigmoid calibration
        return score, {'isolation_score': round(raw, 4), 'z': round(z, 3)}

    # ──────────────────────────────────────────────────────────────────────
    # FIT — build one-class model from APTOS training paths
    # ──────────────────────────────────────────────────────────────────────
    def fit(self, train_paths, sample_n=600, verbose=True):
        """Fit one-class detector on APTOS training images."""
        rng   = np.random.RandomState(42)
        paths = list(train_paths)
        if len(paths) > sample_n:
            idx   = rng.choice(len(paths), sample_n, replace=False)
            paths = [paths[i] for i in idx]

        if verbose:
            print(f'  Extracting features from {len(paths)} APTOS images ...')

        feats, s1_list = [], []
        for p in tqdm(paths, desc='  Verifier.fit', leave=False):
            img = cv2.imread(str(p))
            if img is None: continue
            rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            rgb = cv2.resize(rgb, (self._INF_SIZE, self._INF_SIZE), cv2.INTER_AREA)
            feats.append(self._extract_features(rgb))
            s1, _ = self._stage1_visual(rgb)
            s1_list.append(s1)

        X = np.array(feats, dtype=np.float32)
        self._s1_mu = float(np.mean(s1_list))
        self._s1_sg = float(np.std(s1_list))

        n_comp  = min(self.n_pca, X.shape[0]-1, X.shape[1])
        self.pca = PCA(n_components=n_comp, random_state=42)
        X_pca = self.pca.fit_transform(X)

        self.clf = IsolationForest(
            n_estimators=200, contamination=0.04,
            max_samples=min(256, len(X_pca)),
            random_state=42, n_jobs=-1
        )
        self.clf.fit(X_pca)

        sc = self.clf.score_samples(X_pca)
        self._score_mu = float(sc.mean())
        self._score_sg = float(sc.std())
        self.fitted    = True

        if verbose:
            print(f'  ✅ Fitted | samples={len(X)} | PCA {n_comp}D | '
                  f'IF score μ={self._score_mu:.3f}±{self._score_sg:.3f}')
        return self

    # ──────────────────────────────────────────────────────────────────────
    # VERIFY — full 5-stage pipeline
    # ──────────────────────────────────────────────────────────────────────
    def verify(self, image_input):
        """
        Verify whether image_input is a retinal fundus image.

        Returns
        -------
        dict:
            is_fundus   : bool
            confidence  : float 0–1
            blocked     : bool  (True → stop processing)
            message     : str
            stage_scores: dict
        """
        t0 = time.time()

        # Normalise to uint8 RGB numpy
        try:
            if isinstance(image_input, (str, Path)):
                bgr = cv2.imread(str(image_input))
                if bgr is None: return self._block('Cannot read image file.')
                rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
            elif isinstance(image_input, Image.Image):
                rgb = np.array(image_input.convert('RGB'))
            elif isinstance(image_input, np.ndarray):
                rgb = image_input if image_input.ndim==3 and image_input.shape[2]==3 \
                      else cv2.cvtColor(image_input, cv2.COLOR_GRAY2RGB)
            else:
                return self._block('Unsupported input type.')
        except Exception as e:
            return self._block(f'Read error: {e}')

        # Working copy (256×256)
        rgb_w = cv2.resize(rgb, (256, 256), interpolation=cv2.INTER_AREA)

        # ── Stage 1: Visual (instant) ────────────────────────────────────
        s1, i1 = self._stage1_visual(rgb_w)

        # Fast-reject: completely fails visual — skip expensive stages
        if s1 < 0.10:
            return self._block(
                f'Visual screening failed (s1={s1:.2f}). '
                'Missing fundus colour signature (orange-red hue / green dominance).',
                {'stage1': round(s1,3)})

        # ── Stages 2+3: Vessel topology ──────────────────────────────────
        s23, i23 = self._stage23_vessels(rgb_w)

        # ── Stages 4+5: Deep anomaly ─────────────────────────────────────
        s45, i45 = self._stage45_deep(
            cv2.resize(rgb, (self._INF_SIZE, self._INF_SIZE), cv2.INTER_AREA)
        )

        # ── Weighted fusion ───────────────────────────────────────────────
        if self.fitted:
            # Deep model dominates when fitted
            w = (0.20, 0.30, 0.50)
        else:
            # Fall back to heuristics only
            w = (0.35, 0.65, 0.0)
        confidence = w[0]*s1 + w[1]*s23 + w[2]*s45

        stage_scores = {
            'stage1_visual':   round(s1,  3),
            'stage23_vessels': round(s23, 3),
            'stage45_deep':    round(s45, 3),
            'confidence':      round(confidence, 3),
            **{f's1_{k}': v for k, v in i1.items()},
            **{f's23_{k}': v for k, v in i23.items()},
            **{f's45_{k}': v for k, v in i45.items()},
        }

        is_fundus = confidence >= self.threshold
        ms        = (time.time() - t0) * 1000

        if is_fundus:
            msg = (f'✅ Fundus verified (confidence={confidence:.1%}, '
                   f'vessel={s23:.2f}, deep={s45:.2f}) [{ms:.0f} ms]')
        else:
            msg = (f'🚫 NOT a fundus image (confidence={confidence:.1%} < '
                   f'threshold={self.threshold:.1%}). '
                   f'Vessel score={s23:.2f} | Visual score={s1:.2f}. '
                   f'Please upload a retinal fundus photograph. [{ms:.0f} ms]')

        return {'is_fundus': is_fundus, 'confidence': confidence,
                'blocked': not is_fundus, 'message': msg,
                'stage_scores': stage_scores, 'latency_ms': ms}

    def _block(self, reason, stage_scores=None):
        return {'is_fundus': False, 'confidence': 0.0, 'blocked': True,
                'message': f'🚫 BLOCKED: {reason}',
                'stage_scores': stage_scores or {}, 'latency_ms': 0.0}

    # ── Persistence ───────────────────────────────────────────────────────
    def save(self, path):
        state = {k: getattr(self, k) for k in
                 ('pca','clf','_score_mu','_score_sg','_s1_mu','_s1_sg',
                  'fitted','n_pca','threshold')}
        with open(path, 'wb') as fh: pickle.dump(state, fh)
        print(f'✅ FundusVerifier saved → {path}')

    @classmethod
    def load(cls, path, device=None):
        dev = device or DEVICE
        with open(path, 'rb') as fh: state = pickle.load(fh)
        v = cls(device=dev, n_pca=state['n_pca'], threshold=state['threshold'])
        for k, val in state.items(): setattr(v, k, val)
        return v

print('✅ FundusVerifier class defined.')


## ✂️ Step 9 — Stratified Data Splitting (80 / 10 / 10)

In [ ]:
df_tr, df_te = train_test_split(df, test_size=0.10, stratify=df["diagnosis"], random_state=SEED)
df_tr, df_va = train_test_split(df_tr, test_size=0.10/0.90, stratify=df_tr["diagnosis"], random_state=SEED)

for name, d in [("Train", df_tr), ("Val", df_va), ("Test", df_te)]:
    counts = d["diagnosis"].value_counts().sort_index()
    counts_str = "  ".join([f"G{i}:{v}" for i,v in counts.items()])
    print(f"{name:5s}: {len(d):4d} samples | {counts_str}")

df_tr = df_tr.reset_index(drop=True)
df_va = df_va.reset_index(drop=True)
df_te = df_te.reset_index(drop=True)

# Visualise split distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, (name, d) in zip(axes, [("Train (80%)", df_tr), ("Val (10%)", df_va), ("Test (10%)", df_te)]):
    counts = d["diagnosis"].value_counts().sort_index()
    bars = ax.bar([f"G{i}" for i in counts.index], counts.values,
                   color=GRADE_COLORS, edgecolor="black")
    for bar, v in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()+1,
                str(v), ha="center", fontsize=10, fontweight="bold")
    ax.set_title(f"{name} — {len(d)} samples", fontweight="bold")
    ax.set_xlabel("DR Grade")
    ax.set_ylabel("Count")

plt.suptitle("Stratified Split Distribution", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("/content/split_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Splits created successfully.")


In [ ]:
ARTIFACT_DIR = Path(os.environ.get('ARTIFACT_DIR', '/content/artifacts'))
ARTIFACT_DIR.mkdir(exist_ok=True)

# ── Train the Fundus Verifier on APTOS train split ───────────────────────────
print('=' * 60)
print('🛡️  Training FundusVerifier on APTOS training set')
print('=' * 60)

VERIFIER_CKPT = ARTIFACT_DIR / 'fundus_verifier.pkl'

if VERIFIER_CKPT.exists():
    print('  Loading cached verifier...')
    fundus_verifier = FundusVerifier.load(str(VERIFIER_CKPT))
else:
    fundus_verifier = FundusVerifier(device=DEVICE, n_pca=64, threshold=0.38)
    fundus_verifier.fit(df_tr['image_path'].tolist(), sample_n=600, verbose=True)
    fundus_verifier.save(str(VERIFIER_CKPT))

# ── Smoke-test on val set ─────────────────────────────────────────────────────
print('\n  Smoke-testing on 20 APTOS val images (all should pass):')
_pass = _fail = 0
for _, row in df_va.head(20).iterrows():
    r = fundus_verifier.verify(row['image_path'])
    if r['is_fundus']: _pass += 1
    else: _fail += 1; print(f'  ⚠️  False-reject: {Path(row["image_path"]).name}  {r["message"]}')
print(f'  Pass={_pass}/20  |  False-reject={_fail}/20 (target: 0)')

# ── Stage score distribution on val set ─────────────────────────────────────
print('\n  Stage score stats on 100 APTOS val images:')
_scores = {'s1':[], 's23':[], 's45':[], 'conf':[]}
for _, row in df_va.head(100).iterrows():
    r = fundus_verifier.verify(row['image_path'])
    ss = r['stage_scores']
    _scores['s1'].append(ss.get('stage1_visual',0))
    _scores['s23'].append(ss.get('stage23_vessels',0))
    _scores['s45'].append(ss.get('stage45_deep',0))
    _scores['conf'].append(r['confidence'])
for k, v in _scores.items():
    arr = np.array(v)
    print(f'  {k:6s}: μ={arr.mean():.3f}  σ={arr.std():.3f}  '
          f'min={arr.min():.3f}  max={arr.max():.3f}')

gc.collect()
print('\n✅ FundusVerifier ready.')


## 🔀 Step 10 — Data Augmentation (Train Set Only) & Dataset Class

Medical augmentations: 360° rotation, CLAHE, RandAugment, MixUp.

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# ── RandAugment + Medical augmentations ─────────────────────────────────────
train_transforms = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    # Medical: 360° full rotation support
    A.Rotate(limit=180, p=0.8, border_mode=cv2.BORDER_REFLECT_101),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.15, rotate_limit=45, p=0.5),
    # RandAugment-style: pick 2 from pool
    A.SomeOf([
        A.GaussianBlur(blur_limit=(3, 7), p=1.0),
        A.MedianBlur(blur_limit=5, p=1.0),
        A.MotionBlur(blur_limit=7, p=1.0),
        A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=1.0),
        A.HueSaturationValue(hue_shift_limit=15, sat_shift_limit=30, val_shift_limit=20, p=1.0),
        # Medical CLAHE augmentation
        A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8), p=1.0),
        A.RandomGamma(gamma_limit=(70, 130), p=1.0),
        A.Sharpen(alpha=(0.1, 0.4), lightness=(0.8, 1.2), p=1.0),
    ], n=2, p=0.7),
    A.CoarseDropout(max_holes=8, max_height=IMG_SIZE//16, max_width=IMG_SIZE//16,
                    fill_value=0, p=0.3),
    A.GridDistortion(num_steps=5, distort_limit=0.1, p=0.2),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

val_test_transforms = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

# ── TTA transforms ──────────────────────────────────────────────────────────
tta_transforms = [
    A.Compose([A.Resize(IMG_SIZE, IMG_SIZE),
               A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD), ToTensorV2()]),
    A.Compose([A.Resize(IMG_SIZE, IMG_SIZE), A.HorizontalFlip(p=1.0),
               A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD), ToTensorV2()]),
    A.Compose([A.Resize(IMG_SIZE, IMG_SIZE), A.VerticalFlip(p=1.0),
               A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD), ToTensorV2()]),
    A.Compose([A.Resize(IMG_SIZE, IMG_SIZE), A.Transpose(p=1.0),
               A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD), ToTensorV2()]),
    A.Compose([A.Resize(IMG_SIZE, IMG_SIZE),
               A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=1.0),
               A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD), ToTensorV2()]),
]

# ── MixUp augmentation ───────────────────────────────────────────────────────
def mixup_data(x, y, alpha=0.4):
    if alpha <= 0: return x, y, y, 1.0
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    mixed_x = lam * x + (1 - lam) * x[idx]
    return mixed_x, y, y[idx], lam

def mixup_criterion(criterion, pred, ya, yb, lam):
    return lam * criterion(pred, ya) + (1 - lam) * criterion(pred, yb)

# ── Dataset ──────────────────────────────────────────────────────────────────
class APTOSDataset(Dataset):
    def __init__(self, df, transform=None, use_preprocess=True, cache=False):
        self.df             = df.reset_index(drop=True)
        self.transform      = transform
        self.use_preprocess = use_preprocess
        self._cache         = {} if cache else None

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        if self._cache is not None and idx in self._cache:
            img = self._cache[idx].copy()
        else:
            if self.use_preprocess:
                img = preprocess_fundus(row['image_path'])
                if img is None:
                    img = np.zeros((IMG_SIZE, IMG_SIZE, 3), np.uint8)
            else:
                bgr = cv2.imread(row['image_path'])
                img = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB) if bgr is not None else \
                      np.zeros((IMG_SIZE, IMG_SIZE, 3), np.uint8)
            if self._cache is not None:
                self._cache[idx] = img.copy()
        if self.transform:
            img = self.transform(image=img)['image']
        return img, int(row['diagnosis'])

print('✅ Augmentation pipelines & APTOSDataset defined.')
print(f'  Train transforms: RandAugment (n=2) + 360° rotation + CLAHE')
print(f'  Val/Test: Resize + Normalize only')

## 🧠 Step 11 — Model Architecture (EfficientNetV2-S + Hybrid Head)

**Upgrade:** EfficientNetV2-S backbone (21M params, ImageNet top-1 ~84.9%) with:
- Generalized Mean (GeM) Pooling for richer feature aggregation
- BatchNorm → Dense 512 → SiLU → Dropout → Output head
- Gradient Checkpointing for 1024px VRAM efficiency

In [ ]:
# ── Resolution-aware batch size + gradient accumulation ─────────────────────
if IMG_SIZE >= 1024:
    BATCH_SIZE = 2;  GRAD_ACCUM = 8   # effective batch = 16
elif IMG_SIZE >= 768:
    BATCH_SIZE = 4;  GRAD_ACCUM = 4   # effective batch = 16
elif IMG_SIZE >= 512:
    BATCH_SIZE = 8;  GRAD_ACCUM = 2   # effective batch = 16
else:                                  # 256 or less
    BATCH_SIZE = 16; GRAD_ACCUM = 1

print(f'BATCH_SIZE={BATCH_SIZE}  GRAD_ACCUM={GRAD_ACCUM}  '
      f'(effective batch = {BATCH_SIZE * GRAD_ACCUM})')

# ── Class weights for imbalance handling ────────────────────────────────────
class_counts         = df_tr['diagnosis'].value_counts().sort_index().values
class_weights        = 1.0 / (class_counts / class_counts.sum())
class_weights        = class_weights / class_weights.sum() * len(class_counts)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(DEVICE)
print('Class weights:', {GRADE_MAP[i]: f'{w:.3f}' for i, w in enumerate(class_weights)})

train_ds = APTOSDataset(df_tr, transform=train_transforms)
val_ds   = APTOSDataset(df_va, transform=val_test_transforms)
test_ds  = APTOSDataset(df_te, transform=val_test_transforms)

# ── DataLoader — persistent_workers fixes AssertionError on Colab ────────────
# persistent_workers=True keeps worker processes alive across epochs
# This avoids: 'AssertionError: can only test a child process'
_nw = 2  # 2 workers is safe & fast on Colab T4
train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=_nw, pin_memory=True,
    persistent_workers=True,   # ← KEY: keeps workers alive between epochs
    drop_last=True
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=_nw, pin_memory=True,
    persistent_workers=True
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=_nw, pin_memory=True,
    persistent_workers=True
)

print(f'\nDataLoaders ready (num_workers={_nw}, persistent_workers=True):')
print(f'  Train : {len(train_ds):,} samples ({len(train_loader)} batches)')
print(f'  Val   : {len(val_ds):,}  samples ({len(val_loader)} batches)')
print(f'  Test  : {len(test_ds):,}  samples ({len(test_loader)} batches)')

In [ ]:
NUM_CLASSES = 5   # DR grades 0-4
# ── EfficientNetV2-S: best accuracy/speed for 3662-image dataset ─────────────
BACKBONE = os.environ.get('BACKBONE', 'tf_efficientnetv2_s')  # V2-S >> B4 for accuracy

# ── Generalized Mean Pooling (GeM) ───────────────────────────────────────────
class GeM(nn.Module):
    """GeM pooling — learns optimal pooling exponent p."""
    def __init__(self, p=3, eps=1e-6):
        super().__init__()
        self.p   = nn.Parameter(torch.ones(1) * p)
        self.eps = eps

    def forward(self, x):
        # x: (B, C, H, W)
        return F.avg_pool2d(
            x.clamp(min=self.eps).pow(self.p),
            (x.size(-2), x.size(-1))
        ).pow(1.0 / self.p)

class DRClassifier(nn.Module):
    """
    EfficientNetV2-S backbone + Hybrid Classification Head.
    Upgrades over B0:
      - V2-S Fused-MBConv blocks → faster training
      - GeM pooling → richer feature extraction vs simple AvgPool
      - Gradient checkpointing for 1024px VRAM savings
      - Deeper head: 512-D with SiLU + dual BN
    """
    def __init__(self, backbone=BACKBONE, num_classes=NUM_CLASSES,
                 dropout=0.4, pretrained=True, grad_checkpoint=False):
        super().__init__()
        self.backbone_name = backbone
        self.backbone = timm.create_model(backbone, pretrained=pretrained, num_classes=0,
                                          global_pool=''  # disable timm's pooling
                                         )

        # Gradient checkpointing for high-res (saves ~30% VRAM at 1024px)
        if grad_checkpoint and hasattr(self.backbone, 'set_grad_checkpointing'):
            self.backbone.set_grad_checkpointing(enable=True)
            print('  🧠 Gradient checkpointing ENABLED (high-res mode)')

        feat_dim = self.backbone.num_features

        # GeM Pooling → Flatten
        self.pool = GeM(p=3)

        # Hybrid classification head
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.BatchNorm1d(feat_dim),
            nn.Dropout(dropout),
            nn.Linear(feat_dim, 512),
            nn.SiLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(dropout / 2),
            nn.Linear(512, num_classes)
        )
        for m in self.head.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward_features(self, x):
        feats = self.backbone.forward_features(x)  # (B, C, H, W)
        return self.pool(feats)                     # (B, C, 1, 1)

    def forward(self, x):
        return self.head(self.forward_features(x))

    def get_cam_target_layer(self):
        """Return last conv block for Grad-CAM++."""
        return self.backbone.blocks[-1]

use_grad_ckpt = IMG_SIZE >= 768
model = DRClassifier(backbone=BACKBONE, num_classes=NUM_CLASSES,
                     dropout=0.4, pretrained=True,
                     grad_checkpoint=use_grad_ckpt).to(DEVICE)

total_params   = sum(p.numel() for p in model.parameters())
trainable_tail = sum(p.numel() for p in model.head.parameters())
print(f'Model: {BACKBONE} + GeM Pool + Hybrid Head')
print(f'  Total params:     {total_params/1e6:.2f}M')
print(f'  Head params:      {trainable_tail/1e6:.3f}M')
print(f'  Input resolution: {IMG_SIZE}×{IMG_SIZE}')
print(f'  Mixed Precision:  {DEVICE == "cuda"}')

with torch.no_grad():
    dummy = torch.zeros(2, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
    out   = model(dummy)
    print(f'\n✅ Forward pass OK: {dummy.shape} → {out.shape}')

## ⚖️ Step 12 — Model Compilation (Loss, Optimizer & Scheduler)

In [ ]:
LR           = 2e-4   # slightly lower for V2-S stability
WEIGHT_DECAY = 1e-4
EPOCHS_HEAD  = 5      # Phase 1: head only
EPOCHS_FULL  = 20     # Phase 2: full fine-tune (more epochs for V2-S)
USE_MIXUP    = True
MIXUP_ALPHA  = 0.4

# ── Focal Loss ───────────────────────────────────────────────────────────────
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha     = alpha
        self.gamma     = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce   = F.cross_entropy(inputs, targets, weight=self.alpha, reduction='none')
        pt   = torch.exp(-ce)
        loss = ((1 - pt) ** self.gamma) * ce
        return loss.mean() if self.reduction == 'mean' else loss.sum()

# ── Combined loss: 0.5×CE(label_smooth) + 0.5×Focal ─────────────────────────
ce_criterion    = nn.CrossEntropyLoss(weight=class_weights_tensor, label_smoothing=0.1)
focal_criterion = FocalLoss(alpha=class_weights_tensor, gamma=2.0)

def criterion(logits, labels):
    return 0.5 * ce_criterion(logits, labels) + 0.5 * focal_criterion(logits, labels)

# ── Freeze / unfreeze helpers ─────────────────────────────────────────────────
def freeze_backbone(m):
    for p in m.backbone.parameters(): p.requires_grad_(False)

def unfreeze_backbone(m, unfreeze_blocks=4):
    """Gradually unfreeze last N blocks."""
    for p in m.backbone.parameters(): p.requires_grad_(False)
    blocks = list(m.backbone.blocks)
    for block in blocks[-unfreeze_blocks:]:
        for p in block.parameters(): p.requires_grad_(True)
    # Also unfreeze conv_head / bn2 / norm_head depending on V2 variant
    for attr in ['conv_head', 'bn2', 'norm_head']:
        if hasattr(m.backbone, attr):
            for p in getattr(m.backbone, attr).parameters():
                p.requires_grad_(True)

freeze_backbone(model)

# ── Phase 1 Optimizer + OneCycleLR ───────────────────────────────────────────
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR, weight_decay=WEIGHT_DECAY
)
scaler = GradScaler()

def accuracy_top1(outputs, labels):
    return (outputs.argmax(1) == labels).float().mean().item()

def qwk_score(y_true, y_pred):
    return cohen_kappa_score(y_true, y_pred, weights='quadratic')

print('✅ Loss: 0.5×CE(label_smooth=0.1) + 0.5×Focal(γ=2)')
print(f'   Optimizer : AdamW (lr={LR}, wd={WEIGHT_DECAY})')
print(f'   Scheduler : OneCycleLR (Phase 1) + CosineAnnealingWarmRestarts (Phase 2)')
print(f'   MixUp     : ON (α={MIXUP_ALPHA})')
print(f'   Phase 1   : {EPOCHS_HEAD} epochs (head only)')
print(f'   Phase 2   : {EPOCHS_FULL} epochs (last 4 blocks unfrozen)')
print(f'   Grad accum: {GRAD_ACCUM} steps (effective batch = {BATCH_SIZE * GRAD_ACCUM})')

## 🏋️ Step 13 — Model Training (Phase 1: Head Only — Backbone Frozen)

In [ ]:
ARTIFACT_DIR = Path(os.environ.get('ARTIFACT_DIR', '/content/artifacts'))
ARTIFACT_DIR.mkdir(exist_ok=True)
BEST_CKPT = ARTIFACT_DIR / 'best_model.pt'

history = {'train_loss':[], 'train_acc':[], 'val_loss':[], 'val_acc':[], 'val_qwk':[]}

def run_epoch(loader, training=True, opt=None):
    """Single epoch train/eval with AMP + gradient accumulation."""
    model.train() if training else model.eval()
    total_loss, total_acc = 0., 0.
    all_preds, all_labels = [], []

    _opt = opt if opt is not None else optimizer
    if training: _opt.zero_grad(set_to_none=True)

    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for step, (imgs, labels) in enumerate(
                tqdm(loader, leave=False, desc='Train' if training else 'Val  ')):
            imgs, labels = imgs.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)

            if training:
                if USE_MIXUP:
                    imgs, ya, yb, lam = mixup_data(imgs, labels, alpha=MIXUP_ALPHA)
                with autocast(enabled=(DEVICE=='cuda')):
                    logits = model(imgs)
                    if USE_MIXUP:
                        loss = mixup_criterion(criterion, logits, ya, yb, lam)
                    else:
                        loss = criterion(logits, labels)
                    loss = loss / GRAD_ACCUM

                scaler.scale(loss).backward()

                if (step + 1) % GRAD_ACCUM == 0 or (step + 1) == len(loader):
                    scaler.unscale_(_opt)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(_opt); scaler.update()
                    _opt.zero_grad(set_to_none=True)
                    # Step OneCycleLR per optimizer step (not per epoch)
                    if hasattr(_opt, '_onecycle_sched'):
                        _opt._onecycle_sched.step()

                loss = loss * GRAD_ACCUM
            else:
                with autocast(enabled=(DEVICE=='cuda')):
                    logits = model(imgs)
                    loss   = criterion(logits, labels)

            total_loss += loss.item() * len(imgs)
            total_acc  += accuracy_top1(logits, labels) * len(imgs)
            all_preds.extend(logits.argmax(1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    n = len(loader.dataset)
    return total_loss/n, total_acc/n, qwk_score(all_labels, all_preds)

# ── Phase 1: OneCycleLR — fastest convergence for small datasets ──────────────
# Steps per epoch = ceil(len(train) / (BATCH_SIZE * GRAD_ACCUM))
steps_per_epoch = len(train_loader) // GRAD_ACCUM + 1
scheduler1 = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=LR * 10,            # OneCycleLR peaks at 10× base LR
    steps_per_epoch=steps_per_epoch,
    epochs=EPOCHS_HEAD,
    pct_start=0.3,             # 30% warmup, 70% annealing
    div_factor=25.0,           # start_lr = max_lr / 25
    final_div_factor=1e4,      # end_lr = max_lr / 1e6
    anneal_strategy='cos'
)
# Attach scheduler to optimizer for access inside run_epoch
optimizer._onecycle_sched = scheduler1

print('='*60)
print('PHASE 1 — Training Head Only (Backbone Frozen)')
print(f'Backbone: {BACKBONE} | Resolution: {IMG_SIZE}×{IMG_SIZE}')
print('='*60)

best_val_qwk  = -1.0
best_val_loss = float('inf')
best_epoch    = 0

for epoch in range(1, EPOCHS_HEAD + 1):
    tr_loss, tr_acc, tr_qwk = run_epoch(train_loader, training=True)
    va_loss, va_acc, va_qwk = run_epoch(val_loader,   training=False)

    history['train_loss'].append(tr_loss); history['train_acc'].append(tr_acc)
    history['val_loss'].append(va_loss);   history['val_acc'].append(va_acc)
    history['val_qwk'].append(va_qwk)

    flag = ''
    # Save on best QWK (primary) OR best loss as tiebreak
    if va_qwk > best_val_qwk or (va_qwk == best_val_qwk and va_loss < best_val_loss):
        best_val_qwk = va_qwk; best_val_loss = va_loss; best_epoch = epoch
        torch.save({'epoch': epoch, 'model_state': model.state_dict(),
                    'val_loss': va_loss, 'val_qwk': va_qwk,
                    'history': history}, BEST_CKPT)
        flag = ' ✅ BEST'

    print(f'Ep {epoch:02d}/{EPOCHS_HEAD} | '
          f'TrLoss {tr_loss:.4f} TrAcc {tr_acc:.3f} TrQWK {tr_qwk:.4f} | '
          f'VaLoss {va_loss:.4f} VaAcc {va_acc:.3f} VaQWK {va_qwk:.4f}{flag}')

    gc.collect()
    if DEVICE == 'cuda': torch.cuda.empty_cache()

print(f'\n✅ Phase 1 complete. Best epoch: {best_epoch} | '
      f'Best VaQWK: {best_val_qwk:.4f}')

## 🔓 Step 14 — Fine-Tuning (Backbone Unfreezing + Progressive Resizing)

In [ ]:
print('='*60)
print('PHASE 2 — Full Fine-Tuning (Last 4 Backbone Blocks Unfrozen)')
print('='*60)

# Load best Phase 1 checkpoint
ckpt = torch.load(BEST_CKPT, map_location=DEVICE)
model.load_state_dict(ckpt['model_state'])

unfreeze_backbone(model, unfreeze_blocks=4)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable params now: {trainable/1e6:.2f}M')

# ── Progressive Resizing Schedule ────────────────────────────────────────────
# Gradually increase resolution: better generalization & faster early training
PROG_SIZES   = [256, 512, IMG_SIZE]  # phases: warmup → main → full-res
PROG_EPOCHS  = [4, 8, EPOCHS_FULL - 12]  # epochs per size (sum = EPOCHS_FULL)
if EPOCHS_FULL <= 10:
    PROG_SIZES  = [512, IMG_SIZE]
    PROG_EPOCHS = [EPOCHS_FULL // 2, EPOCHS_FULL - EPOCHS_FULL // 2]

print(f'Progressive resizing: {list(zip(PROG_SIZES, PROG_EPOCHS))}')

# ── Differential LR: backbone gets 10× lower LR than head ────────────────────
optimizer2 = torch.optim.AdamW([
    {'params': model.head.parameters(),                    'lr': LR / 5},
    {'params': model.pool.parameters(),                    'lr': LR / 5},
    {'params': [p for n, p in model.backbone.named_parameters()
                if p.requires_grad],                       'lr': LR / 50},
], weight_decay=WEIGHT_DECAY)

# ── CosineAnnealingWarmRestarts for Phase 2 ───────────────────────────────────
scheduler2 = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer2, T_0=max(EPOCHS_FULL // 3, 5), T_mult=1, eta_min=1e-7)

patience, no_improve = 8, 0
epoch_global = 0

for prog_idx, (res_size, n_ep) in enumerate(zip(PROG_SIZES, PROG_EPOCHS)):
    if n_ep <= 0: continue
    print(f'\n--- Progressive Resize Phase {prog_idx+1}: {res_size}×{res_size}, {n_ep} epochs ---')

    # Rebuild dataloaders at new resolution if needed
    if res_size != IMG_SIZE:
        _tr = A.Compose([A.Resize(res_size, res_size),
                         A.Rotate(limit=180, p=0.7, border_mode=cv2.BORDER_REFLECT_101),
                         A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5),
                         A.RandomBrightnessContrast(0.25, 0.25, p=0.5),
                         A.CLAHE(clip_limit=3.0, p=0.4),
                         A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD), ToTensorV2()])
        _vt = A.Compose([A.Resize(res_size, res_size),
                         A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD), ToTensorV2()])
        _tr_ds = APTOSDataset(df_tr, transform=_tr, use_preprocess=False)
        _va_ds = APTOSDataset(df_va, transform=_vt, use_preprocess=False)
        _bs = max(2, BATCH_SIZE * (IMG_SIZE // res_size) ** 2)
        _tr_ld = DataLoader(_tr_ds, batch_size=min(_bs, 32), shuffle=True,
                            num_workers=2, pin_memory=True, persistent_workers=True,
                            drop_last=True)
        _va_ld = DataLoader(_va_ds, batch_size=min(_bs, 32), shuffle=False,
                            num_workers=2, pin_memory=True, persistent_workers=True)
    else:
        _tr_ld, _va_ld = train_loader, val_loader

    for epoch in range(1, n_ep + 1):
        epoch_global += 1
        tr_loss, tr_acc, tr_qwk = run_epoch(_tr_ld, training=True, opt=optimizer2)
        va_loss, va_acc, va_qwk = run_epoch(_va_ld, training=False)
        scheduler2.step(epoch_global)

        history['train_loss'].append(tr_loss); history['train_acc'].append(tr_acc)
        history['val_loss'].append(va_loss);   history['val_acc'].append(va_acc)
        history['val_qwk'].append(va_qwk)

        flag = ''
        if va_qwk > best_val_qwk or (va_qwk == best_val_qwk and va_loss < best_val_loss):
            best_val_qwk = va_qwk; best_val_loss = va_loss
            best_epoch   = EPOCHS_HEAD + epoch_global
            no_improve   = 0
            torch.save({'epoch': best_epoch, 'model_state': model.state_dict(),
                        'val_loss': va_loss, 'val_qwk': va_qwk,
                        'history': history}, BEST_CKPT)
            flag = ' ✅ BEST'
        else:
            no_improve += 1
            flag = f' ({no_improve}/{patience})'

        print(f'Ep {epoch_global:02d}/{EPOCHS_FULL} [{res_size}px] | '
              f'TrLoss {tr_loss:.4f} TrAcc {tr_acc:.3f} | '
              f'VaLoss {va_loss:.4f} VaAcc {va_acc:.3f} QWK {va_qwk:.4f}{flag}')

        gc.collect()
        if DEVICE == 'cuda': torch.cuda.empty_cache()

        if no_improve >= patience:
            print(f'\n⏹️  Early stopping triggered.')
            break
    else:
        continue
    break

print(f'\n✅ Phase 2 complete. Best epoch: {best_epoch} | Best VaQWK: {best_val_qwk:.4f}')

## 📈 Step 15 — Training Curves & Learning Rate Schedule

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
epochs_range = range(1, len(history["train_loss"]) + 1)
phase_line   = EPOCHS_HEAD + 0.5  # vertical divider between phases

for ax, (title, tr_key, va_key) in zip(axes, [
    ("Loss",      "train_loss", "val_loss"),
    ("Accuracy",  "train_acc",  "val_acc"),
    ("Val QWK",   None,         "val_qwk"),
]):
    if tr_key:
        ax.plot(epochs_range, history[tr_key], "b-o", ms=4, label="Train")
    ax.plot(epochs_range, history[va_key], "r-o", ms=4, label="Val")
    ax.axvline(phase_line, color="grey", linestyle="--", alpha=0.7, label="Fine-tune start")
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Epoch"); ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle("Training History", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(str(ARTIFACT_DIR / "training_curves.png"), dpi=150, bbox_inches="tight")
plt.show()

# Print best metrics
best_va_qwk = max(history["val_qwk"])
best_va_acc = max(history["val_acc"])
print(f"Best Val QWK:      {best_va_qwk:.4f}")
print(f"Best Val Accuracy: {best_va_acc:.4f} ({best_va_acc*100:.2f}%)")


## 📋 Step 16 — Evaluation (Confusion Matrix, QWK, AUROC, Sensitivity/Specificity)

In [ ]:
# Load best checkpoint
ckpt = torch.load(BEST_CKPT, map_location=DEVICE)
model.load_state_dict(ckpt["model_state"])
model.eval()
print(f"✅ Loaded best checkpoint (epoch {ckpt['epoch']}, val_loss {ckpt['val_loss']:.4f})")

def get_predictions(loader, use_tta=True):
    """
    Run inference with optional Test-Time Augmentation (TTA).
    TTA averages predictions over 5 augmented views for higher accuracy.
    """
    all_logits, all_labels = [], []

    with torch.no_grad():
        for imgs, labels in tqdm(loader, desc="Evaluating"):
            imgs = imgs.to(DEVICE)
            if use_tta:
                # Re-load raw images for TTA (transforms applied manually)
                # Fast path: average logits from base + 4 TTA transforms
                batch_logits = []
                with autocast(enabled=(DEVICE=="cuda")):
                    batch_logits.append(model(imgs))
                    # Horizontal flip
                    batch_logits.append(model(torch.flip(imgs, [-1])))
                    # Vertical flip
                    batch_logits.append(model(torch.flip(imgs, [-2])))
                    # Both flips
                    batch_logits.append(model(torch.flip(imgs, [-1, -2])))
                logits = torch.stack(batch_logits).mean(0)
            else:
                with autocast(enabled=(DEVICE=="cuda")):
                    logits = model(imgs)

            all_logits.append(logits.cpu())
            all_labels.extend(labels.numpy())

    logits_cat = torch.cat(all_logits, 0)
    probs_cat  = F.softmax(logits_cat, dim=1).numpy()
    preds_cat  = logits_cat.argmax(1).numpy()
    return probs_cat, preds_cat, np.array(all_labels)

print("\n── Validation set (with TTA) ──")
va_probs, va_preds, va_labels = get_predictions(val_loader,  use_tta=True)
print("── Test set (with TTA) ──")
te_probs, te_preds, te_labels = get_predictions(test_loader, use_tta=True)

In [ ]:
# ── Confusion Matrix ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
grade_names = [GRADE_MAP[i] for i in range(5)]

for ax, (name, labels, preds) in zip(axes, [
    ("Validation", va_labels, va_preds),
    ("Test",       te_labels, te_preds),
]):
    cm = confusion_matrix(labels, preds)
    cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
    im = ax.imshow(cm_pct, cmap="Blues", vmin=0, vmax=100)
    ax.set_xticks(range(5)); ax.set_yticks(range(5))
    ax.set_xticklabels([f"G{i}" for i in range(5)], rotation=30, ha="right")
    ax.set_yticklabels([f"G{i}" for i in range(5)])
    for i in range(5):
        for j in range(5):
            ax.text(j, i, f"{cm[i,j]}\n({cm_pct[i,j]:.0f}%)",
                    ha="center", va="center",
                    color="white" if cm_pct[i,j] > 50 else "black",
                    fontsize=9)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(f"{name} Confusion Matrix", fontweight="bold")
    plt.colorbar(im, ax=ax, label="Row %")

plt.tight_layout()
plt.savefig(str(ARTIFACT_DIR / "confusion_matrix.png"), dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ── Metrics Summary ──────────────────────────────────────────────────────────
from sklearn.preprocessing import label_binarize

def compute_metrics(labels, probs, preds, split=""):
    n_cls = probs.shape[1]
    y_bin = label_binarize(labels, classes=list(range(n_cls)))

    # OvR AUROC / AUPRC
    auroc = roc_auc_score(y_bin, probs, multi_class="ovr", average="macro")
    auprc = average_precision_score(y_bin, probs, average="macro")
    qwk   = qwk_score(labels, preds)
    acc   = (labels == preds).mean()

    # Binary (referable DR) metrics
    bin_true  = (labels >= 2).astype(int)
    bin_prob  = probs[:, 2:].sum(axis=1)  # P(grade ≥ 2)
    bin_auroc = roc_auc_score(bin_true, bin_prob)
    bin_auprc = average_precision_score(bin_true, bin_prob)

    # Threshold at spec ≈ 0.95
    from sklearn.metrics import confusion_matrix as cm_fn
    best_t, best_diff = 0.5, 1e9
    for t in np.linspace(0, 1, 1001):
        yhat = (bin_prob >= t).astype(int)
        tn, fp, fn, tp = cm_fn(bin_true, yhat).ravel()
        d = abs(tn/(tn+fp+1e-9) - 0.95)
        if d < best_diff:
            best_t, best_diff = t, d
    yhat_opt = (bin_prob >= best_t).astype(int)
    tn, fp, fn, tp = cm_fn(bin_true, yhat_opt).ravel()
    spec = tn / (tn + fp + 1e-9)
    sens = tp / (tp + fn + 1e-9)

    print(f"\n{'='*50}")
    print(f"  {split} Set Metrics")
    print(f"{'='*50}")
    print(f"  Multi-class Accuracy:  {acc*100:.2f}%")
    print(f"  Quadratic Weighted K:  {qwk:.4f}")
    print(f"  OvR AUROC (macro):     {auroc:.4f}")
    print(f"  OvR AUPRC (macro):     {auprc:.4f}")
    print(f"  Binary AUROC:          {bin_auroc:.4f}")
    print(f"  Binary AUPRC:          {bin_auprc:.4f}")
    print(f"  Threshold @ Spec≈0.95: {best_t:.3f}")
    print(f"  Specificity:           {spec:.4f}")
    print(f"  Sensitivity:           {sens:.4f}")
    print(f"{'='*50}")

    return dict(split=split, acc=acc, qwk=qwk, auroc=auroc, auprc=auprc,
                bin_auroc=bin_auroc, bin_auprc=bin_auprc,
                threshold=best_t, spec=spec, sens=sens)

val_metrics  = compute_metrics(va_labels, va_probs, va_preds, "Validation")
test_metrics = compute_metrics(te_labels, te_probs, te_preds, "Test")

# Save CSV
pd.DataFrame([val_metrics, test_metrics]).to_csv(
    str(ARTIFACT_DIR / "metrics_summary.csv"), index=False)
print("\n✅ Metrics saved to artifacts/metrics_summary.csv")


In [ ]:
# ── Classification Report ────────────────────────────────────────────────────
print("\n── Test Set Per-Class Report ──")
print(classification_report(
    te_labels, te_preds,
    target_names=[f"G{i}: {GRADE_MAP[i]}" for i in range(5)],
    digits=3
))

# ── ROC Curves ───────────────────────────────────────────────────────────────
from sklearn.metrics import roc_curve
y_bin_te = label_binarize(te_labels, classes=list(range(5)))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for i in range(5):
    fpr, tpr, _ = roc_curve(y_bin_te[:, i], te_probs[:, i])
    auc_i = roc_auc_score(y_bin_te[:, i], te_probs[:, i])
    axes[0].plot(fpr, tpr, label=f"G{i} {GRADE_MAP[i]} (AUC={auc_i:.3f})",
                  color=GRADE_COLORS[i], linewidth=2)
axes[0].plot([0,1],[0,1], "k--", alpha=0.4)
axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR")
axes[0].set_title("ROC Curves (Test — One-vs-Rest)", fontweight="bold")
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

# Binary PR curve
from sklearn.metrics import precision_recall_curve
bin_true_te = (te_labels >= 2).astype(int)
bin_prob_te = te_probs[:, 2:].sum(axis=1)
prec, rec, _ = precision_recall_curve(bin_true_te, bin_prob_te)
axes[1].plot(rec, prec, "r-", linewidth=2)
axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
axes[1].set_title(f"Binary PR Curve (AUPRC={test_metrics['bin_auprc']:.4f})", fontweight="bold")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(str(ARTIFACT_DIR / "roc_pr_curves.png"), dpi=150, bbox_inches="tight")
plt.show()


## 💾 Step 17 — Save Model

In [ ]:
FINAL_CKPT = ARTIFACT_DIR / "dr_classifier_final.pt"

torch.save({
    "model_state":   model.state_dict(),
    "backbone":      BACKBONE,          # ← was: efficientnet_b0 (NameError fixed)
    "num_classes":   NUM_CLASSES,
    "img_size":      IMG_SIZE,
    "grade_map":     GRADE_MAP,
    "val_metrics":   val_metrics,
    "test_metrics":  test_metrics,
    "history":       history,
    "seed":          SEED,
}, FINAL_CKPT)

print(f"✅ Model saved: {FINAL_CKPT}")
print(f"   Size: {FINAL_CKPT.stat().st_size / 1e6:.1f} MB")

# Optional download in Colab
if IN_COLAB:
    # Uncomment to download:
    # colab_files.download(str(FINAL_CKPT))
    pass
else:
    print(f"   Download from: {FINAL_CKPT}")

## ✅ Step 18 — Input Image Validation

In [ ]:
SUPPORTED_FORMATS = {'.png', '.jpg', '.jpeg'}
MAX_FILE_SIZE_MB  = 15.0    # increased for high-res fundus images
MIN_DIM_PX        = 64
MAX_DIM_PX        = 8192

def validate_image(file_path_or_bytes):
    """
    Two-tier image validation:
      Tier 1 — format/size/dimension checks (fast)
      Tier 2 — FundusVerifier multilayer check (blocks non-fundus)

    Returns: (is_valid: bool, message: str, pil_image: PIL.Image | None)
    """
    try:
        if isinstance(file_path_or_bytes, (bytes, bytearray)):
            size_mb = len(file_path_or_bytes) / 1e6
            img     = Image.open(io.BytesIO(file_path_or_bytes))
            ext     = (img.format or 'unknown').lower()
        else:
            p   = Path(file_path_or_bytes)
            ext = p.suffix.lower()
            if ext not in SUPPORTED_FORMATS:
                return False, f'❌ Unsupported format "{ext}". Use PNG or JPG.', None
            size_mb = p.stat().st_size / 1e6
            img     = Image.open(p)

        if size_mb > MAX_FILE_SIZE_MB:
            return False, f'❌ File too large ({size_mb:.1f} MB > {MAX_FILE_SIZE_MB} MB).', None

        img = img.convert('RGB')
        w, h = img.size
        if w < MIN_DIM_PX or h < MIN_DIM_PX:
            return False, f'❌ Image too small ({w}×{h} px).', None
        if w > MAX_DIM_PX or h > MAX_DIM_PX:
            return False, f'❌ Image too large ({w}×{h} px).', None

        # ── Tier 2: Multilayer Fundus Verifier ───────────────────────────
        vr = fundus_verifier.verify(img)
        if vr['blocked']:
            return False, vr['message'], None

        return True, (
            f'✅ Valid fundus image ({w}×{h} px, {size_mb:.2f} MB). '
            f'{vr["message"]}'
        ), img

    except Exception as e:
        return False, f'❌ Could not read image: {e}', None

# ── Quick test ────────────────────────────────────────────────────────────────
test_path = df_te['image_path'].iloc[0]
_ok, _msg, _img = validate_image(test_path)
print(f'Test image: {Path(test_path).name}')
print(f'Result: {_ok} | {_msg}')


## 🔮 Step 19 — Prediction Function

In [ ]:
def predict(image_input, top_k=5):
    """
    Predict DR grade from a fundus image.
    Raises ValueError if FundusVerifier blocks the image.

    Parameters
    ----------
    image_input : str | Path | PIL.Image | np.ndarray
    Returns dict: grade, grade_label, confidence, probabilities,
                  preprocessed_img, verification
    """
    model.eval()

    # ── Gate 1: Fundus Verifier ───────────────────────────────────────────
    vr = fundus_verifier.verify(image_input)
    if vr['blocked']:
        raise ValueError(
            f'FundusVerifier BLOCKED: {vr["message"]}\n'
            f'Scores: {vr["stage_scores"]}')

    # ── Load + preprocess image ───────────────────────────────────────────
    if isinstance(image_input, (str, Path)):
        valid, msg, pil_img = validate_image(image_input)
        if not valid or pil_img is None:
            raise ValueError(msg)
        arr = np.array(pil_img)
        bgr = cv2.cvtColor(arr, cv2.COLOR_RGB2BGR)
        tmp = '/tmp/_dr_input.png'; cv2.imwrite(tmp, bgr)
        preprocessed = preprocess_fundus(tmp)
    elif isinstance(image_input, Image.Image):
        arr = np.array(image_input.convert('RGB'))
        bgr = cv2.cvtColor(arr, cv2.COLOR_RGB2BGR)
        tmp = '/tmp/_dr_input.png'; cv2.imwrite(tmp, bgr)
        preprocessed = preprocess_fundus(tmp)
    elif isinstance(image_input, np.ndarray):
        if image_input.dtype != np.uint8:
            image_input = (image_input * 255).clip(0, 255).astype(np.uint8)
        bgr = cv2.cvtColor(image_input, cv2.COLOR_RGB2BGR) \
              if image_input.ndim==3 else image_input
        tmp = '/tmp/_dr_input.png'; cv2.imwrite(tmp, bgr)
        preprocessed = preprocess_fundus(tmp)
    else:
        raise TypeError(f'Unsupported type: {type(image_input)}')

    if preprocessed is None:
        raise ValueError('Preprocessing failed — invalid image.')

    # ── Transform + infer ─────────────────────────────────────────────────
    aug    = val_test_transforms(image=preprocessed)
    tensor = aug['image'].unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        with autocast(enabled=(DEVICE=='cuda')):
            logits = model(tensor)
    probs = F.softmax(logits, dim=1)[0].cpu().numpy()

    grade = int(probs.argmax()); confidence = float(probs[grade])
    return {
        'grade':           grade,
        'grade_label':     GRADE_MAP[grade],
        'confidence':      confidence,
        'probabilities':   {GRADE_MAP[i]: float(probs[i]) for i in range(NUM_CLASSES)},
        'preprocessed_img': preprocessed,
        'verification':    vr,
    }

# ── Batch test on 5 samples ───────────────────────────────────────────────────
print('🔍 Testing predict() on 5 samples:')
for i in range(5):
    row = df_te.iloc[i]
    try:
        res = predict(row['image_path'])
        ok  = '✅' if res['grade'] == row['diagnosis'] else '❌'
        print(f'  {ok} True:G{row["diagnosis"]} Pred:G{res["grade"]} '
              f'({res["confidence"]*100:.1f}%) verify={res["verification"]["confidence"]:.2f}')
    except ValueError as e:
        print(f'  ⚠️  Blocked: {e}')


## 🗺️ Step 20 — Explainability with Grad-CAM++

In [ ]:
# ── Grad-CAM++ wrapper ────────────────────────────────────────────────────────
def generate_gradcam(image_input, target_class=None):
    """
    Generate Grad-CAM++ heatmap overlay for a given image.

    Parameters
    ----------
    image_input  : path | PIL.Image | np.ndarray
    target_class : int | None  (None → predicted class)

    Returns
    -------
    (original_rgb, overlay_rgb, predicted_result_dict)
    """
    result = predict(image_input)
    tc = target_class if target_class is not None else result["grade"]

    # ── Prepare input tensor ─────────────────────────────────────────────────
    preprocessed = result["preprocessed_img"]
    aug    = val_test_transforms(image=preprocessed)
    tensor = aug["image"].unsqueeze(0).to(DEVICE)

    # ── Run GradCAM++ ────────────────────────────────────────────────────────
    target_layer = [model.get_cam_target_layer()]

    with GradCAMPlusPlus(model=model, target_layers=target_layer) as cam:
        from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
        grayscale_cam = cam(
            input_tensor=tensor,
            targets=[ClassifierOutputTarget(tc)],
            eigen_smooth=True,
            aug_smooth=True
        )[0]  # (H, W)

    # ── Overlay ───────────────────────────────────────────────────────────────
    # Normalize preprocessed image to [0,1] float for overlay
    orig_float = preprocessed.astype(np.float32) / 255.0
    overlay = show_cam_on_image(orig_float, grayscale_cam, use_rgb=True, image_weight=0.5)

    return preprocessed, overlay, grayscale_cam, result

# ── Test on one sample ────────────────────────────────────────────────────────
sample_path = df_va.iloc[5]["image_path"]
orig, overlay, heatmap, result = generate_gradcam(sample_path)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(orig);     axes[0].set_title("Preprocessed Fundus", fontweight="bold")
axes[1].imshow(overlay);  axes[1].set_title(f"Grad-CAM++ Overlay\n(Target: Grade {result['grade']})", fontweight="bold")
axes[2].imshow(heatmap, cmap="jet"); axes[2].set_title("Raw Heatmap", fontweight="bold")
for ax in axes: ax.axis("off")
plt.suptitle(f"Grad-CAM++ | True: G{df_va.iloc[5]['diagnosis']} | "
             f"Pred: G{result['grade']} {result['grade_label']} "
             f"({result['confidence']*100:.1f}%)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(str(ARTIFACT_DIR / "sample_gradcam.png"), dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ── Grad-CAM++ Board: 3 per grade ────────────────────────────────────────────
fig, axes = plt.subplots(5, 6, figsize=(22, 18))

for grade in range(5):
    grade_rows = df_va[df_va["diagnosis"] == grade]
    samples    = grade_rows.head(3)

    for j, (_, row) in enumerate(samples.iterrows()):
        orig, overlay, _, result = generate_gradcam(row["image_path"])

        col_orig    = j * 2
        col_overlay = j * 2 + 1

        axes[grade][col_orig].imshow(orig)
        axes[grade][col_orig].axis("off")
        axes[grade][col_orig].set_title(f"Original", fontsize=8)

        pred_correct = "✅" if result["grade"] == grade else "❌"
        axes[grade][col_overlay].imshow(overlay)
        axes[grade][col_overlay].axis("off")
        axes[grade][col_overlay].set_title(
            f"CAM {pred_correct} Pred:G{result['grade']} ({result['confidence']*100:.0f}%)",
            fontsize=7.5, color="green" if result["grade"]==grade else "red")

    # Row label
    axes[grade][0].set_ylabel(f"G{grade}\n{GRADE_MAP[grade]}",
                               fontsize=9, fontweight="bold",
                               color=GRADE_COLORS[grade],
                               rotation=0, labelpad=85, va="center")

plt.suptitle("Grad-CAM++ Explainability Board\n"
             "Original Fundus (left) vs CAM Overlay (right) — All DR Grades",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(str(ARTIFACT_DIR / "gradcam_board.png"), dpi=120, bbox_inches="tight")
plt.show()
print("✅ Grad-CAM++ board saved.")


## 🚀 Step 21 — Interactive Deployment (Gradio UI)

> A full web interface with:
> - File upload (PNG / JPG, max 5 MB)
> - Real-time DR grade prediction
> - Grad-CAM++ explainability overlay
> - 5-class confidence bar chart
> - Severity description & clinical context


In [ ]:
import gradio as gr
import matplotlib
matplotlib.use('Agg')

SEVERITY_INFO = {
    0: {'emoji':'🟢','color':'#2ecc71','label':'No DR',
        'detail':'No signs of diabetic retinopathy.','action':'Annual follow-up.'},
    1: {'emoji':'🟡','color':'#f1c40f','label':'Mild DR',
        'detail':'Microaneurysms only.','action':'Refer within 12 months.'},
    2: {'emoji':'🟠','color':'#e67e22','label':'Moderate DR',
        'detail':'More than microaneurysms; possible exudates.','action':'Refer 3–6 months. ⚠️'},
    3: {'emoji':'🔴','color':'#e74c3c','label':'Severe DR',
        'detail':'Extensive hemorrhages, venous beading, IRMA.','action':'Urgent referral 4 wks. 🚨'},
    4: {'emoji':'🟣','color':'#8e44ad','label':'Proliferative DR',
        'detail':'Neovascularization present.','action':'Emergency referral. 🆘'},
}

def make_prob_chart(probabilities):
    fig, ax = plt.subplots(figsize=(7, 3.2))
    grades = list(probabilities.keys())
    values = [v*100 for v in probabilities.values()]
    bars   = ax.barh(grades, values, color=GRADE_COLORS, edgecolor='white', height=0.55)
    for bar, val in zip(bars, values):
        ax.text(min(val+1, 96), bar.get_y()+bar.get_height()/2,
                f'{val:.1f}%', va='center', fontsize=11, fontweight='bold', color='#333')
    ax.set_xlim(0, 105); ax.set_xlabel('Confidence (%)', fontsize=11)
    ax.set_title('Prediction Confidence by DR Grade', fontsize=12,
                 fontweight='bold', pad=10)
    ax.spines[['top','right']].set_visible(False)
    ax.grid(axis='x', alpha=0.3); ax.invert_yaxis()
    plt.tight_layout(); return fig

def make_verifier_badge(vr):
    """Render a compact verifier score card as HTML."""
    ss   = vr.get('stage_scores', {})
    conf = vr.get('confidence', 0)
    ok   = vr.get('is_fundus', False)
    col  = '#27ae60' if ok else '#e74c3c'
    icon = '✅' if ok else '🚫'
    bars = ''
    stage_labels = [
        ('stage1_visual',   '1 Visual'),
        ('stage23_vessels', '2 Vessels'),
        ('stage45_deep',    '3 Deep AI'),
    ]
    for key, lbl in stage_labels:
        sc  = ss.get(key, 0)
        pct = sc * 100
        c   = '#27ae60' if sc >= 0.4 else '#e67e22' if sc >= 0.2 else '#e74c3c'
        bars += f"""
          <div style='display:flex;align-items:center;gap:8px;margin:3px 0;font-size:0.8rem;'>
            <span style='width:70px;color:#555;'>{lbl}</span>
            <div style='flex:1;background:#eee;border-radius:4px;height:12px;'>
              <div style='width:{pct:.0f}%;background:{c};border-radius:4px;height:12px;'></div>
            </div>
            <span style='width:36px;text-align:right;color:{c};font-weight:700;'>{pct:.0f}%</span>
          </div>"""
    return f"""
<div style='font-family:sans-serif;border:2px solid {col};border-radius:10px;
            padding:12px;background:{col}11;margin-bottom:8px;'>
  <div style='font-size:1rem;font-weight:800;color:{col};margin-bottom:8px;'>
    {icon} Fundus Verifier — {'PASS' if ok else 'BLOCKED'} ({conf:.1%})</div>
  {bars}
</div>"""

def predict_ui(image):
    if image is None:
        return None, None, None, '⚠️ No image provided.', '', ''
    try:
        pil_img = Image.fromarray(image.astype(np.uint8)) \
                  if isinstance(image, np.ndarray) else image
        arr     = np.array(pil_img)

        # ── Stage 0: Run verifier FIRST ───────────────────────────────────
        vr = fundus_verifier.verify(arr)
        verifier_html = make_verifier_badge(vr)

        if vr['blocked']:
            blocked_html = f"""
<div style='font-family:sans-serif;background:#fff3cd;border:2px solid #ffc107;
            border-radius:10px;padding:20px;'>
  <div style='font-size:1.3rem;font-weight:800;color:#856404;'>🚫 Non-Fundus Image Detected</div>
  <div style='margin-top:8px;color:#664d03;'>{vr['message']}</div>
  <hr style='border:1px solid #ffc10755;margin:10px 0;'/>
  <div style='font-size:0.85rem;color:#664d03;'>
    <b>This image has been blocked from processing.</b><br/>
    Please upload a retinal fundus photograph (colour fundus camera image).
  </div>
</div>"""
            return None, None, None, blocked_html, verifier_html, ''

        # ── Grade prediction (verifier passed) ────────────────────────────
        orig, overlay, hm, result = generate_gradcam(arr)
        grade = result['grade']; conf = result['confidence']*100
        info  = SEVERITY_INFO[grade]
        chart = make_prob_chart(result['probabilities'])

        result_html = f"""
<div style='font-family:sans-serif;padding:16px;border-radius:12px;
            background:{info['color']}22;border:2px solid {info['color']};'>
  <div style='font-size:2rem;'>{info['emoji']}</div>
  <div style='font-size:1.5rem;font-weight:800;color:{info['color']};'>
    Grade {grade} — {info['label']}</div>
  <div style='font-weight:700;margin:6px 0;'>Confidence: {conf:.1f}%</div>
  <hr style='border:1px solid {info['color']}55;margin:10px 0;'/>
  <div style='color:#444;'><b>Finding:</b> {info['detail']}</div>
  <div style='color:#444;'><b>Action:</b> {info['action']}</div>
</div>
<div style='font-size:0.75rem;color:#888;font-style:italic;padding:4px 0;'>
  ⚠️ Research tool only. Not a substitute for clinical diagnosis.</div>"""

        val_html = f'<div style="font-size:0.85rem;color:#555;">✔ {result["verification"]["message"]}</div>'

        return overlay, orig, chart, result_html, verifier_html, val_html

    except ValueError as ve:
        err = f"""<div style='background:#fff3cd;border:1px solid #ffc107;
            border-radius:8px;padding:16px;color:#856404;'>
  <b>🚫 Blocked:</b><br/>{str(ve)}</div>"""
        return None, None, None, err, '', ''
    except Exception as e:
        err = f"""<div style='background:#f8d7da;border:1px solid #f5c6cb;
            border-radius:8px;padding:16px;color:#721c24;'>
  <b>⚠️ Error:</b><br/>{str(e)}</div>"""
        return None, None, None, err, '', ''

CUSTOM_CSS = """
body, .gradio-container { font-family: 'Segoe UI', sans-serif !important; }
.gradio-container { max-width: 1300px !important; margin: auto !important; }
footer { display: none !important; }
"""

with gr.Blocks(css=CUSTOM_CSS, title='DR Grading System',
               theme=gr.themes.Soft()) as demo:
    gr.HTML("""
    <div style='text-align:center;padding:20px 0 10px;'>
      <h1 style='font-size:1.9rem;font-weight:800;color:#1a1a2e;'>
        🩺 AI-Powered Diabetic Retinopathy Grading System</h1>
      <p style='color:#555;'>EfficientNet-B4 + Grad-CAM++ | 5-Stage Fundus Verifier | APTOS 2019</p>
      <p style='color:#e74c3c;font-size:0.85rem;font-style:italic;'>
        ⚠️ Research use only. Not for clinical deployment.</p>
    </div>""")

    with gr.Row():
        with gr.Column(scale=1):
            image_input  = gr.Image(type='numpy', label='📤 Upload Fundus Image', height=300)
            predict_btn  = gr.Button('🔬 Analyze Image', variant='primary', size='lg')
            verifier_out = gr.HTML('<div style="color:#888;font-size:0.85rem;">Fundus verifier result will appear here.</div>')
            val_out      = gr.HTML()

        with gr.Column(scale=2):
            with gr.Row():
                orig_out = gr.Image(type='numpy', label='🖼️ Preprocessed', height=270)
                cam_out  = gr.Image(type='numpy', label='🗺️ Grad-CAM++',   height=270)
            chart_out   = gr.Plot(label='📊 Confidence Scores')
            result_display = gr.HTML('<div style="color:#aaa;text-align:center;padding:20px;">Result will appear here.</div>')

    gr.HTML('<hr style="margin:20px 0;"/>')
    gr.Markdown('### 🧪 Example Images')
    with gr.Row():
        example_imgs = []
        for grade in range(5):
            ex = df_te[df_te['diagnosis']==grade]
            if len(ex):
                example_imgs.append([ex.iloc[0]['image_path']])
                gr.Image(value=ex.iloc[0]['image_path'],
                         label=f'G{grade}: {GRADE_MAP[grade]}',
                         height=110, interactive=False)
    gr.Examples(examples=example_imgs, inputs=image_input, label='Click to load')

    _outs = [cam_out, orig_out, chart_out, result_display, verifier_out, val_out]
    predict_btn.click(fn=predict_ui, inputs=image_input, outputs=_outs)
    image_input.change(fn=predict_ui, inputs=image_input, outputs=_outs)

# ── Launch (Render-compatible — reads PORT env var) ───────────────────────────
PORT        = int(os.environ.get('PORT', 7860))
SERVER_NAME = '0.0.0.0'
print(f'✅ Launching on {SERVER_NAME}:{PORT}  (ngrok removed — use Render for public URL)')
demo.launch(server_name=SERVER_NAME, server_port=PORT, share=False,
            debug=False, quiet=True)


## 🌐 Step 22 — Deploy to Render

The next cell writes all files needed to host this app on **Render.com** (free tier).

| File | Purpose |
|------|----------|
| `app.py` | Standalone Gradio server (loads saved model) |
| `requirements.txt` | Python dependencies for Render |
| `render.yaml` | Infrastructure-as-code config |

**Steps:**
1. Run this cell to generate the files.
2. Push everything to a GitHub repo.
3. On [render.com](https://render.com) → **New Web Service** → connect the repo.
4. Render auto-detects `render.yaml` and deploys. Your URL is `https://<service>.onrender.com`.


In [ ]:
# ── Generate Render deployment artefacts ─────────────────────────────────────
deploy_dir = Path('/content/render_deploy')
deploy_dir.mkdir(exist_ok=True)

# Copy verifier pkl into deploy directory
import shutil as _sh
if VERIFIER_CKPT.exists():
    _sh.copy2(VERIFIER_CKPT, deploy_dir / 'fundus_verifier.pkl')
    print(f'✅ Copied verifier → {deploy_dir / "fundus_verifier.pkl"}')

# 1) requirements.txt
(deploy_dir / 'requirements.txt').write_text(
'torch==2.2.2+cpu ; sys_platform=="linux"\n'
'torchvision==0.17.2+cpu ; sys_platform=="linux"\n'
'timm>=0.9.0\n'
'albumentations>=1.3.1\n'
'opencv-python-headless>=4.9.0\n'
'scikit-learn>=1.3.0\n'
'scikit-image>=0.21.0\n'
'scipy>=1.11.0\n'
'matplotlib>=3.7.0\n'
'tqdm>=4.66.0\n'
'grad-cam>=1.5.2\n'
'gradio>=4.0.0\n'
'gunicorn>=21.2.0\n'
'pyyaml>=6.0.1\n'
'numpy>=1.24.0\n'
'pandas>=2.0.0\n'
'Pillow>=10.0.0\n'
)

# 2) render.yaml
(deploy_dir / 'render.yaml').write_text(
'services:\n'
'  - type: web\n'
'    name: dr-grading-app\n'
'    runtime: python\n'
'    plan: free\n'
'    buildCommand: pip install -r requirements.txt\n'
'    startCommand: python app.py\n'
'    healthCheckPath: /\n'
'    envVars:\n'
'      - key: PORT\n'
'        value: 10000\n'
'      - key: IMG_SIZE\n'
'        value: 512\n'
'      - key: BACKBONE\n'
'        value: efficientnet_b4\n'
'      - key: MODEL_PATH\n'
'        value: /opt/render/project/src/dr_classifier_final.pt\n'
'      - key: VERIFIER_PATH\n'
'        value: /opt/render/project/src/fundus_verifier.pkl\n'
)

# 3) app.py — self-contained Render server with FundusVerifier
APP_PY = r'''
"""DR Grading — Render deployment (Gradio + FundusVerifier)."""
import os, io, gc, time, warnings, pickle
from pathlib import Path
import numpy as np, cv2, torch, torch.nn as nn, torch.nn.functional as F
import timm, albumentations as A
from albumentations.pytorch import ToTensorV2
from PIL import Image
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import gradio as gr
from pytorch_grad_cam import GradCAMPlusPlus
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest

warnings.filterwarnings("ignore")

IMG_SIZE    = int(os.environ.get("IMG_SIZE",    512))
BACKBONE    = os.environ.get("BACKBONE",        "efficientnet_b4")
MODEL_PATH  = os.environ.get("MODEL_PATH",      "dr_classifier_final.pt")
VERIFIER_PATH = os.environ.get("VERIFIER_PATH", "fundus_verifier.pkl")
PORT        = int(os.environ.get("PORT",         10000))
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
NUM_CLASSES = 5
GRADE_MAP    = {0:"No DR",1:"Mild DR",2:"Moderate DR",3:"Severe DR",4:"Proliferative DR"}
GRADE_COLORS = ["#2ecc71","#f1c40f","#e67e22","#e74c3c","#8e44ad"]
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# ── DR Model ──────────────────────────────────────────────────────────────────
class DRClassifier(nn.Module):
    def __init__(self, backbone=BACKBONE, num_classes=NUM_CLASSES, dropout=0.4):
        super().__init__()
        self.backbone = timm.create_model(backbone, pretrained=False, num_classes=0)
        fd = self.backbone.num_features
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.BatchNorm1d(fd), nn.Dropout(dropout),
            nn.Linear(fd, 512), nn.SiLU(),
            nn.BatchNorm1d(512), nn.Dropout(dropout/2),
            nn.Linear(512, num_classes)
        )
    def forward_features(self, x): return self.backbone.forward_features(x)
    def forward(self, x):          return self.head(self.forward_features(x))
    def get_cam_target_layer(self): return self.backbone.blocks[-1]

model = DRClassifier().to(DEVICE)
if Path(MODEL_PATH).exists():
    ckpt = torch.load(MODEL_PATH, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state"])
    print(f"DR model loaded from {MODEL_PATH}")
else:
    print(f"WARNING: model not found at {MODEL_PATH} (UI only)")
model.eval()

# ── Preprocessing ─────────────────────────────────────────────────────────────
_SIG = max((IMG_SIZE // 10) | 1, 1)
def _mask(rgb):
    g=rgb[:,:,1]; gb=cv2.medianBlur(g,7)
    _,th=cv2.threshold(gb,0,255,cv2.THRESH_BINARY+cv2.THRESH_OTSU)
    th=cv2.morphologyEx(th,cv2.MORPH_OPEN,cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(7,7)))
    cnts,_=cv2.findContours(th,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
    if not cnts: return np.ones(g.shape,np.uint8)*255
    c=max(cnts,key=cv2.contourArea); mask=np.zeros_like(g,np.uint8)
    cv2.drawContours(mask,[c],-1,255,-1)
    (cx,cy),r=cv2.minEnclosingCircle(c); circ=np.zeros_like(g,np.uint8)
    cv2.circle(circ,(int(cx),int(cy)),int(r*.97),255,-1)
    return cv2.bitwise_and(mask,circ)
def preprocess_fundus(arr_rgb, size=IMG_SIZE):
    h,w=arr_rgb.shape[:2]; s=size/max(h,w)
    nh,nw=int(round(h*s)),int(round(w*s))
    rgb=cv2.resize(arr_rgb,(nw,nh),interpolation=cv2.INTER_AREA)
    pt=(size-nh)//2;pb=size-nh-pt;pl=(size-nw)//2;pr=size-nw-pl
    rgb=cv2.copyMakeBorder(rgb,pt,pb,pl,pr,cv2.BORDER_REFLECT_101)
    mask=_mask(rgb); rgb[mask==0]=0
    lab=cv2.cvtColor(rgb,cv2.COLOR_RGB2LAB); l,a,b=cv2.split(lab)
    l2=cv2.createCLAHE(2.0,(8,8)).apply(l)
    rgb=cv2.cvtColor(cv2.merge([l2,a,b]),cv2.COLOR_LAB2RGB)
    blur=cv2.GaussianBlur(rgb,(0,0),sigmaX=_SIG)
    rgb=cv2.addWeighted(rgb,4,blur,-4,128); rgb[mask==0]=0
    r2,g2,b2=rgb[:,:,0].astype(np.float32),rgb[:,:,1].astype(np.float32),rgb[:,:,2].astype(np.float32)
    mix=(r2*.1+g2*.8+b2*.1).clip(0,255).astype(np.uint8)
    return np.stack([mix,rgb[:,:,1],mix],2)

INF_TFM = A.Compose([A.Resize(IMG_SIZE,IMG_SIZE),
    A.Normalize(mean=IMAGENET_MEAN,std=IMAGENET_STD),ToTensorV2()])

# ── FundusVerifier (portable class embedded in app.py) ────────────────────────
class FundusVerifier:
    _INF_SIZE=224; _MEAN=[.485,.456,.406]; _STD=[.229,.224,.225]
    def __init__(self,device=DEVICE,n_pca=64,threshold=0.38):
        self.device=device; self.n_pca=n_pca; self.threshold=threshold; self.fitted=False
        self._extractor=timm.create_model('mobilenetv3_small_050',pretrained=True,num_classes=0).to(device).eval()
        self._tfm=A.Compose([A.Resize(self._INF_SIZE,self._INF_SIZE),
            A.Normalize(mean=self._MEAN,std=self._STD),ToTensorV2()])
        self.pca=self.clf=None; self._score_mu=self._score_sg=0.0
    def _feat(self,rgb):
        t=self._tfm(image=rgb)["image"].unsqueeze(0).to(self.device)
        with torch.no_grad(): return self._extractor(t).cpu().numpy().flatten()
    def _s1(self,rgb):
        a=rgb.astype(np.float32); g=a[:,:,1]; r=a[:,:,0]; b=a[:,:,2]
        gd=float(g.mean()/(r.mean()+b.mean()+1e-6))
        hsv=cv2.cvtColor(rgb,cv2.COLOR_RGB2HSV).astype(np.float32)
        vm=hsv[:,:,2]>20; hv=hsv[:,:,0][vm]
        orr=float(((hv<30)|(hv>155)).mean()) if vm.sum()>100 else 0.0
        bright=float(8<g.mean()<235); dark=float((g<12).mean())
        return min(gd/1.4,1.0)*.3+orr*.3+bright*.2+float(dark<.8)*.2
    def _s23(self,rgb):
        sm=cv2.resize(rgb,(256,256),cv2.INTER_AREA); g=sm[:,:,1]
        ge=cv2.createCLAHE(3.0,(8,8)).apply(g)
        vr=np.zeros_like(ge,np.float32)
        for ks in[7,11,15,21]:
            k=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(ks,ks))
            vr=np.maximum(vr,cv2.morphologyEx(ge,cv2.MORPH_BLACKHAT,k).astype(np.float32))
        mn,mx=vr.min(),vr.max()
        vn=(vr-mn)/(mx-mn) if mx>mn else vr
        vb=(vn>0.18).astype(np.uint8)*255
        vd=float(vb.mean())/255; mr=float(vn.mean())
        n,_,st,_=cv2.connectedComponentsWithStats(vb,connectivity=8)
        ar=st[1:,cv2.CC_STAT_AREA] if n>1 else np.array([])
        lc=int((ar>=20).sum()) if len(ar) else 0
        qs=[float(vb[r0:r1,c0:c1].mean())/255 for r0,r1,c0,c1 in
            [(0,128,0,128),(0,128,128,256),(128,256,0,128),(128,256,128,256)]]
        nq=sum(q>0.005 for q in qs)
        return float(0.005<vd<0.20)*.25+min(mr/.05,1.)*.25+min(lc/4.,1.)*.25+nq/4.*.25
    def _s45(self,rgb):
        if not self.fitted: return 0.5
        xp=self.pca.transform(self._feat(rgb).reshape(1,-1))
        raw=float(self.clf.score_samples(xp)[0])
        z=(raw-self._score_mu)/(self._score_sg+1e-6)
        return float(1/(1+np.exp(-z*2.5)))
    def verify(self,image_input):
        try:
            if isinstance(image_input,np.ndarray): rgb=image_input
            elif isinstance(image_input,Image.Image): rgb=np.array(image_input.convert("RGB"))
            else: return {"is_fundus":False,"blocked":True,"confidence":0,"message":"Bad input","stage_scores":{}}
            rw=cv2.resize(rgb,(256,256),cv2.INTER_AREA)
            s1=self._s1(rw)
            if s1<0.10: return {"is_fundus":False,"blocked":True,"confidence":s1,
                "message":f"🚫 Rejected: visual fail (s1={s1:.2f})",
                "stage_scores":{"stage1_visual":round(s1,3)}}
            s23=self._s23(rw)
            ri=cv2.resize(rgb,(self._INF_SIZE,self._INF_SIZE),cv2.INTER_AREA)
            s45=self._s45(ri)
            w=(0.20,0.30,0.50) if self.fitted else (0.35,0.65,0.0)
            conf=w[0]*s1+w[1]*s23+w[2]*s45
            ok=conf>=self.threshold
            msg=(f"✅ Verified ({conf:.1%})" if ok else
                 f"🚫 NOT fundus (conf={conf:.1%}). Upload retinal fundus photo.")
            return {"is_fundus":ok,"blocked":not ok,"confidence":conf,"message":msg,
                    "stage_scores":{"stage1_visual":round(s1,3),
                        "stage23_vessels":round(s23,3),"stage45_deep":round(s45,3)}}
        except Exception as e:
            return {"is_fundus":False,"blocked":True,"confidence":0,
                    "message":f"🚫 Error: {e}","stage_scores":{}}
    @classmethod
    def load(cls,path,device=None):
        dev=device or DEVICE
        with open(path,"rb") as fh: st=pickle.load(fh)
        v=cls(device=dev,n_pca=st["n_pca"],threshold=st["threshold"])
        for k,val in st.items(): setattr(v,k,val)
        return v

# Load verifier
if Path(VERIFIER_PATH).exists():
    fundus_verifier = FundusVerifier.load(VERIFIER_PATH)
    print(f"FundusVerifier loaded from {VERIFIER_PATH}")
else:
    print("WARNING: verifier not found, using heuristics only")
    fundus_verifier = FundusVerifier()

# ── Inference helpers ─────────────────────────────────────────────────────────
SEVERITY_INFO = {
    0:{"emoji":"🟢","color":"#2ecc71","label":"No DR",
       "detail":"No signs of DR.","action":"Annual follow-up."},
    1:{"emoji":"🟡","color":"#f1c40f","label":"Mild DR",
       "detail":"Microaneurysms only.","action":"Refer 12 months."},
    2:{"emoji":"🟠","color":"#e67e22","label":"Moderate DR",
       "detail":"Exudates/hemorrhages.","action":"Refer 3-6 months ⚠️"},
    3:{"emoji":"🔴","color":"#e74c3c","label":"Severe DR",
       "detail":"IRMA/beading.","action":"Urgent 4 weeks 🚨"},
    4:{"emoji":"🟣","color":"#8e44ad","label":"Proliferative DR",
       "detail":"Neovascularization.","action":"Emergency 🆘"},
}

def predict(arr_rgb):
    vr = fundus_verifier.verify(arr_rgb)
    if vr["blocked"]: raise ValueError(vr["message"])
    prep = preprocess_fundus(arr_rgb)
    t = INF_TFM(image=prep)["image"].unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        probs = F.softmax(model(t),1)[0].cpu().numpy()
    g = int(probs.argmax())
    return {"grade":g,"grade_label":GRADE_MAP[g],"confidence":float(probs[g]),
            "probabilities":{GRADE_MAP[i]:float(probs[i]) for i in range(NUM_CLASSES)},
            "preprocessed_img":prep, "verification":vr}

def gradcam(arr_rgb):
    res = predict(arr_rgb); prep = res["preprocessed_img"]
    t = INF_TFM(image=prep)["image"].unsqueeze(0).to(DEVICE)
    with GradCAMPlusPlus(model=model,target_layers=[model.get_cam_target_layer()]) as cam:
        gcam = cam(t,[ClassifierOutputTarget(res["grade"])],
                   eigen_smooth=True,aug_smooth=True)[0]
    ov = show_cam_on_image(prep.astype(np.float32)/255., gcam, use_rgb=True, image_weight=0.5)
    return prep, ov, res

def make_chart(probs):
    fig,ax=plt.subplots(figsize=(7,3.2))
    g=list(probs.keys()); v=[x*100 for x in probs.values()]
    bars=ax.barh(g,v,color=GRADE_COLORS,edgecolor="white",height=0.55)
    for bar,val in zip(bars,v):
        ax.text(min(val+1,96),bar.get_y()+bar.get_height()/2,
                f"{val:.1f}%",va="center",fontsize=11,fontweight="bold",color="#333")
    ax.set_xlim(0,105); ax.invert_yaxis()
    ax.spines[["top","right"]].set_visible(False); ax.grid(axis="x",alpha=0.3)
    plt.tight_layout(); return fig

def ui_predict(image):
    if image is None: return None,None,None,"Upload image.","",""
    try:
        arr = image if isinstance(image,np.ndarray) else np.array(image)
        vr  = fundus_verifier.verify(arr)
        ss  = vr.get("stage_scores",{})
        vhtml = f"""<div style='border:2px solid {'#27ae60' if vr['is_fundus'] else '#e74c3c'};
            border-radius:10px;padding:12px;font-family:sans-serif;'>
          <b>{'✅ Fundus Verified' if vr['is_fundus'] else '🚫 BLOCKED'}</b> ({vr['confidence']:.1%})
          <div>S1={ss.get('stage1_visual',0):.2f} | S23={ss.get('stage23_vessels',0):.2f} | S45={ss.get('stage45_deep',0):.2f}</div>
        </div>"""
        if vr["blocked"]:
            return None,None,None,f"<div style='background:#fff3cd;padding:16px;border-radius:8px;'><b>🚫 {vr['message']}</b></div>",vhtml,""
        orig,ov,res = gradcam(arr)
        gr_=res["grade"]; c=res["confidence"]*100; inf=SEVERITY_INFO[gr_]
        rhtml=f"""<div style='font-family:sans-serif;padding:16px;border-radius:12px;
            background:{inf['color']}22;border:2px solid {inf['color']};'>
          <div style='font-size:1.4rem;font-weight:800;color:{inf['color']};'>
            {inf['emoji']} Grade {gr_} — {inf['label']}</div>
          <div style='font-weight:700;'>Confidence: {c:.1f}%</div>
          <hr/><div><b>Finding:</b> {inf['detail']}</div>
          <div><b>Action:</b> {inf['action']}</div></div>"""
        return ov,orig,make_chart(res["probabilities"]),rhtml,vhtml,"✅ OK"
    except Exception as e:
        return None,None,None,f"<b>Error:</b> {e}","",""

with gr.Blocks(title="DR Grading",theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🩺 DR Grading — EfficientNet-B4 + 5-Stage Fundus Verifier")
    gr.Markdown("> ⚠️ Research use only.")
    with gr.Row():
        with gr.Column(scale=1):
            inp     = gr.Image(type="numpy",label="Upload Fundus Image")
            btn     = gr.Button("🔬 Analyze",variant="primary")
            v_out   = gr.HTML(label="Verifier")
            val_out = gr.HTML()
        with gr.Column(scale=2):
            with gr.Row():
                orig_o = gr.Image(type="numpy",label="Preprocessed")
                cam_o  = gr.Image(type="numpy",label="Grad-CAM++")
            chart_o  = gr.Plot(label="Confidence")
            res_o    = gr.HTML()
    btn.click(ui_predict,inp,[cam_o,orig_o,chart_o,res_o,v_out,val_out])
    inp.change(ui_predict,inp,[cam_o,orig_o,chart_o,res_o,v_out,val_out])

if __name__=="__main__":
    print(f"Starting on 0.0.0.0:{PORT}")
    demo.launch(server_name="0.0.0.0",server_port=PORT,share=False)
'''.strip()

(deploy_dir / 'app.py').write_text(APP_PY)

print('✅ Render deployment files written:')
for f in sorted(deploy_dir.iterdir()):
    print(f'  {f.name}  ({f.stat().st_size:,} bytes)')
print('\n📋 Deploy steps:')
print('  1. Copy dr_classifier_final.pt + fundus_verifier.pkl into repo root')
print('  2. Push app.py, requirements.txt, render.yaml to GitHub')
print('  3. render.com → New Web Service → connect repo')
print('  4. Render reads render.yaml → builds → deploys')
print('  5. Live URL: https://<service-name>.onrender.com')


## 📝 Summary & Results

In [ ]:
print('=' * 65)
print('  DIABETIC RETINOPATHY GRADING — FINAL RESULTS SUMMARY')
print('=' * 65)

for split, m in [('Validation', val_metrics), ('Test', test_metrics)]:
    print(f'\n  {split} Set:')
    print(f'    Accuracy:          {m["acc"]*100:.2f}%')
    print(f'    Quadratic WK:      {m["qwk"]:.4f}')
    print(f'    Binary AUROC:      {m["bin_auroc"]:.4f}')
    print(f'    Binary AUPRC:      {m["bin_auprc"]:.4f}')
    print(f'    Sensitivity:       {m["sens"]:.4f}  (@thr={m["threshold"]:.3f})')
    print(f'    Specificity:       {m["spec"]:.4f}')

print('\n  Artifacts saved:')
for f in sorted(ARTIFACT_DIR.glob('*')):
    print(f'    {f.name}  ({f.stat().st_size/1e3:.1f} KB)')

print(f'\n  Model: {BACKBONE} + GeM Pool + Hybrid Head (timm)')
print('  Dataset: APTOS 2019 Blindness Detection (Kaggle)')
print('  Upgrades: EfficientNetV2-S | OneCycleLR | Progressive Resize')
print('           persistent_workers=True | RandAugment | GeM Pool')
print('  Explainability: Grad-CAM++ (pytorch-grad-cam)')
print('  Deployment: Gradio (share link above)')
print('=' * 65)
print('  ⚠️  RESEARCH USE ONLY — NOT FOR CLINICAL DEPLOYMENT')
print('=' * 65)